# Exploratory hybrid anatomy-aware classifier training

**Public role:** Public exploratory model-development notebook; not part of the required article reproduction path.

**Relation to the manuscript:** Retained for transparency. The hybrid candidate was not used for the final external AQPR claims or the frozen article classifier panel.

**Protocol boundary:** This notebook is an exploratory development branch. It is published for research transparency but is excluded from the final article reproduction sequence and must not be interpreted as HYGD-driven selection of the frozen article classifier panel.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `11_hybrid_anatomy_aware_classifier_training.ipynb`


In [ ]:
# Cell 1: Setup for hybrid anatomy-aware classifier training

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import random
import platform
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass


def safe_file_exists(path):
    try:
        return Path(path).exists()
    except OSError:
        return False
    except Exception:
        return False


def non_empty_path_series(series):
    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return ~normalized.isin(["", "nan", "none", "null"])


def parse_bool_series(series):
    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "y": True,
                "valid": True,
                "false": False,
                "0": False,
                "no": False,
                "n": False,
                "nan": False,
                "none": False,
                "": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))

PROJECT_DATA_DIR = PROJECT_ROOT / "project_data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_07_DIR = RESULTS_DIR / "07_hygd_glaucoma_classifier_benchmark"
RESULTS_08_DIR = RESULTS_DIR / "08_anatomical_explainability_validation"
RESULTS_09_DIR = RESULTS_DIR / "09_anatomy_ablation_and_shortcut_controls"
RESULTS_10_DIR = RESULTS_DIR / "10_anatomy_constrained_classifier_training"
RESULTS_11_DIR = RESULTS_DIR / "11_hybrid_anatomy_aware_classifier_training"

RESULTS_11_DIR.mkdir(parents=True, exist_ok=True)

HYBRID_DATA_DIR = (
    PROJECT_DATA_DIR
    / "hygd_hybrid_anatomy_aware_classifier_inputs"
)

HYBRID_DATA_DIR.mkdir(parents=True, exist_ok=True)

HYBRID_MODELS_DIR = (
    MODELS_DIR
    / "hygd_hybrid_anatomy_aware_classifiers"
)

HYBRID_MODELS_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK07_FINAL_STATUS_JSON = RESULTS_07_DIR / "notebook07_final_status.json"
NOTEBOOK10_FINAL_STATUS_JSON = RESULTS_10_DIR / "notebook10_final_status.json"

ANALYSIS_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_final_od_oc_proxy"
    / "hygd_final_od_oc_proxy_analysis_manifest.csv"
)

SPLIT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_glaucoma_classifier_splits"
    / "hygd_image_level_split_manifest_seed42.csv"
)

ANATOMY_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_anatomy_constrained_classifier_inputs"
    / "notebook10_strict_anatomy_constrained_input_manifest.csv"
)

NOTEBOOK10_FINAL_SYNTHESIS_CSV = (
    RESULTS_10_DIR
    / "notebook10_final_synthesis.csv"
)

NOTEBOOK10_FINAL_MODEL_ROLES_CSV = (
    RESULTS_10_DIR
    / "notebook10_final_model_roles.csv"
)

HYBRID_PAIRED_INPUT_MANIFEST_CSV = (
    HYBRID_DATA_DIR
    / "notebook11_hybrid_full_image_od25x_paired_input_manifest.csv"
)

HYBRID_EXPERIMENT_PLAN_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_experiment_plan.csv"
)

HYBRID_EXPERIMENT_PLAN_JSON = (
    RESULTS_11_DIR
    / "notebook11_hybrid_experiment_plan.json"
)

INPUT_AUDIT_CSV = (
    RESULTS_11_DIR
    / "notebook11_input_artifact_audit.csv"
)

SETUP_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_setup_summary.json"
)

required_inputs = [
    NOTEBOOK07_FINAL_STATUS_JSON,
    NOTEBOOK10_FINAL_STATUS_JSON,
    ANALYSIS_MANIFEST_CSV,
    SPLIT_MANIFEST_CSV,
    ANATOMY_INPUT_MANIFEST_CSV,
    NOTEBOOK10_FINAL_SYNTHESIS_CSV,
    NOTEBOOK10_FINAL_MODEL_ROLES_CSV,
]

input_audit = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "path": str(path),
            "exists": safe_file_exists(path),
        }
        for path in required_inputs
    ]
)

input_audit.to_csv(INPUT_AUDIT_CSV, index=False)

missing_inputs = input_audit.loc[
    ~input_audit["exists"],
    "path",
].tolist()

if missing_inputs:
    print(input_audit.to_string(index=False))
    raise FileNotFoundError(f"Missing required Notebook 11 inputs: {missing_inputs}")

with open(NOTEBOOK07_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook07_status = json.load(f)

with open(NOTEBOOK10_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook10_status = json.load(f)

analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)
split_manifest = pd.read_csv(SPLIT_MANIFEST_CSV)
anatomy_input_manifest = pd.read_csv(ANATOMY_INPUT_MANIFEST_CSV)
notebook10_synthesis = pd.read_csv(NOTEBOOK10_FINAL_SYNTHESIS_CSV)
notebook10_model_roles = pd.read_csv(NOTEBOOK10_FINAL_MODEL_ROLES_CSV)

required_analysis_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "image_path_for_inference",
    "quality_score",
    "area_cdr_proxy",
]

missing_analysis_cols = [
    col for col in required_analysis_cols
    if col not in analysis_manifest.columns
]

if missing_analysis_cols:
    raise ValueError(f"Missing analysis manifest columns: {missing_analysis_cols}")

required_split_cols = [
    "sample_id",
    "patient_id",
    "glaucoma_split",
]

missing_split_cols = [
    col for col in required_split_cols
    if col not in split_manifest.columns
]

if missing_split_cols:
    raise ValueError(f"Missing split manifest columns: {missing_split_cols}")

required_anatomy_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "gon_label",
    "glaucoma_split",
    "experiment_id",
    "constrained_image_path",
]

missing_anatomy_cols = [
    col for col in required_anatomy_cols
    if col not in anatomy_input_manifest.columns
]

if missing_anatomy_cols:
    raise ValueError(f"Missing anatomy input manifest columns: {missing_anatomy_cols}")

analysis_manifest["sample_id"] = analysis_manifest["sample_id"].astype(str)
analysis_manifest["patient_id"] = analysis_manifest["patient_id"].astype(str)
analysis_manifest["gon_binary"] = pd.to_numeric(
    analysis_manifest["gon_binary"],
    errors="coerce",
).astype(int)

split_manifest["sample_id"] = split_manifest["sample_id"].astype(str)
split_manifest["patient_id"] = split_manifest["patient_id"].astype(str)
split_manifest["glaucoma_split"] = split_manifest["glaucoma_split"].astype(str)

anatomy_input_manifest["sample_id"] = anatomy_input_manifest["sample_id"].astype(str)
anatomy_input_manifest["patient_id"] = anatomy_input_manifest["patient_id"].astype(str)
anatomy_input_manifest["experiment_id"] = anatomy_input_manifest["experiment_id"].astype(str)
anatomy_input_manifest["glaucoma_split"] = anatomy_input_manifest["glaucoma_split"].astype(str)
anatomy_input_manifest["gon_binary"] = pd.to_numeric(
    anatomy_input_manifest["gon_binary"],
    errors="coerce",
).astype(int)

if "gon_label" not in analysis_manifest.columns:
    analysis_manifest["gon_label"] = np.where(
        analysis_manifest["gon_binary"] == 1,
        "GON+",
        "GON-",
    )
else:
    analysis_manifest["gon_label"] = analysis_manifest["gon_label"].astype(str)

od25_manifest = anatomy_input_manifest[
    anatomy_input_manifest["experiment_id"] == "od_centered_crop_2p5x"
].copy()

if len(od25_manifest) == 0:
    raise RuntimeError("OD-centered 2.5x anatomy input manifest is empty.")

base_manifest = analysis_manifest[
    [
        "sample_id",
        "patient_id",
        "gon_binary",
        "gon_label",
        "image_path_for_inference",
        "quality_score",
        "area_cdr_proxy",
    ]
].copy()

split_block = split_manifest[
    [
        "sample_id",
        "patient_id",
        "glaucoma_split",
    ]
].drop_duplicates().copy()

base_manifest = base_manifest.merge(
    split_block,
    on=["sample_id", "patient_id"],
    how="left",
    validate="one_to_one",
)

if base_manifest["glaucoma_split"].isna().any():
    missing_split = int(base_manifest["glaucoma_split"].isna().sum())
    raise RuntimeError(f"Missing split assignment in base manifest: {missing_split}")

od25_manifest = od25_manifest[
    [
        "sample_id",
        "patient_id",
        "experiment_id",
        "constrained_image_path",
        "crop_scale",
        "context_policy",
        "background_policy",
    ]
].copy()

hybrid_manifest = base_manifest.merge(
    od25_manifest,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

hybrid_manifest = hybrid_manifest.rename(
    columns={
        "image_path_for_inference": "full_image_path",
        "constrained_image_path": "od_centered_2p5x_image_path",
    }
)

hybrid_manifest["full_image_path"] = hybrid_manifest["full_image_path"].astype(str)
hybrid_manifest["od_centered_2p5x_image_path"] = hybrid_manifest["od_centered_2p5x_image_path"].astype(str)

hybrid_manifest["full_image_path_non_empty"] = non_empty_path_series(
    hybrid_manifest["full_image_path"]
)

hybrid_manifest["od_centered_2p5x_path_non_empty"] = non_empty_path_series(
    hybrid_manifest["od_centered_2p5x_image_path"]
)

hybrid_manifest = hybrid_manifest[
    hybrid_manifest["full_image_path_non_empty"]
    & hybrid_manifest["od_centered_2p5x_path_non_empty"]
].copy().reset_index(drop=True)

if len(hybrid_manifest) == 0:
    raise RuntimeError("Hybrid paired manifest is empty.")

hybrid_manifest["hybrid_experiment_id"] = "hybrid_full_image_plus_od_centered_2p5x_effb0"
hybrid_manifest["full_branch_input"] = "original_full_image"
hybrid_manifest["anatomy_branch_input"] = "od_centered_crop_2p5x"
hybrid_manifest["full_branch_backbone"] = "efficientnet_b0"
hybrid_manifest["anatomy_branch_backbone"] = "efficientnet_b0"
hybrid_manifest["fusion_strategy"] = "late_feature_concatenation"

hybrid_manifest.to_csv(HYBRID_PAIRED_INPUT_MANIFEST_CSV, index=False)

split_summary = (
    hybrid_manifest
    .groupby("glaucoma_split")
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        n_positive=("gon_binary", "sum"),
        mean_quality_score=("quality_score", "mean"),
        mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
    )
    .reset_index()
)

split_summary["n_negative"] = (
    split_summary["n_images"]
    - split_summary["n_positive"]
)

patient_overlap_rows = []

split_names = sorted(hybrid_manifest["glaucoma_split"].unique().tolist())

for idx_a, split_a in enumerate(split_names):
    patients_a = set(
        hybrid_manifest.loc[
            hybrid_manifest["glaucoma_split"] == split_a,
            "patient_id",
        ].astype(str)
    )

    for split_b in split_names[idx_a + 1:]:
        patients_b = set(
            hybrid_manifest.loc[
                hybrid_manifest["glaucoma_split"] == split_b,
                "patient_id",
            ].astype(str)
        )

        patient_overlap_rows.append(
            {
                "split_a": split_a,
                "split_b": split_b,
                "n_overlap_patients": len(patients_a.intersection(patients_b)),
            }
        )

patient_overlap_df = pd.DataFrame(patient_overlap_rows)

hybrid_experiment_plan = pd.DataFrame(
    [
        {
            "hybrid_experiment_id": "hybrid_full_image_plus_od_centered_2p5x_effb0",
            "training_role": "primary_hybrid_anatomy_aware_classifier",
            "full_branch_input": "original_full_image",
            "anatomy_branch_input": "od_centered_crop_2p5x",
            "full_branch_backbone": "efficientnet_b0",
            "anatomy_branch_backbone": "efficientnet_b0",
            "fusion_strategy": "late_feature_concatenation",
            "fusion_head": "concat_features_dropout_mlp_binary_head",
            "input_size": 384,
            "selection_metric": "patient_val_auroc_then_brier",
            "primary_comparators": (
                "full_image__EfficientNetB0; anatomy__od_centered_crop_2p5x; "
                "anatomy__od_centered_crop_4p0x; anatomy__outside_od_only_trained_control"
            ),
            "journal_role": (
                "Test whether global full-image context and explicit OD-centered anatomical focus "
                "can be combined while preserving performance and improving interpretability."
            ),
        }
    ]
)

hybrid_experiment_plan.to_csv(HYBRID_EXPERIMENT_PLAN_CSV, index=False)

with open(HYBRID_EXPERIMENT_PLAN_JSON, "w", encoding="utf-8") as f:
    json.dump(
        hybrid_experiment_plan.to_dict(orient="records"),
        f,
        indent=2,
    )

setup_summary = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "status": "PASS",
    "objective": (
        "Train a hybrid anatomy-aware glaucoma classifier using paired full-image and OD-centered 2.5x inputs. "
        "The model will test whether global context and explicit anatomical focus can be fused while preserving performance."
    ),
    "device": str(DEVICE),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "project_root": str(PROJECT_ROOT),
    "notebook07_status": notebook07_status.get("status", "unknown"),
    "notebook10_status": notebook10_status.get("status", "unknown"),
    "notebook10_primary_interpretation": notebook10_status.get("primary_interpretation", ""),
    "total_hybrid_rows": int(len(hybrid_manifest)),
    "total_hybrid_patients": int(hybrid_manifest["patient_id"].nunique()),
    "split_summary": split_summary.to_dict(orient="records"),
    "patient_overlap": patient_overlap_df.to_dict(orient="records"),
    "primary_hybrid_experiment": "hybrid_full_image_plus_od_centered_2p5x_effb0",
    "input_audit_csv": str(INPUT_AUDIT_CSV),
    "hybrid_paired_input_manifest_csv": str(HYBRID_PAIRED_INPUT_MANIFEST_CSV),
    "hybrid_experiment_plan_csv": str(HYBRID_EXPERIMENT_PLAN_CSV),
    "hybrid_experiment_plan_json": str(HYBRID_EXPERIMENT_PLAN_JSON),
}

with open(SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)

print(f"Device: {DEVICE}")
print(f"Notebook 07 status: {setup_summary['notebook07_status']}")
print(f"Notebook 10 status: {setup_summary['notebook10_status']}")
print(f"Total hybrid rows: {setup_summary['total_hybrid_rows']}")
print(f"Total hybrid patients: {setup_summary['total_hybrid_patients']}")

print("\nHybrid split summary:")
print(split_summary.to_string(index=False))

print("\nPatient overlap check:")
print(patient_overlap_df.to_string(index=False))

print("\nHybrid experiment plan:")
print(
    hybrid_experiment_plan[
        [
            "hybrid_experiment_id",
            "full_branch_input",
            "anatomy_branch_input",
            "full_branch_backbone",
            "anatomy_branch_backbone",
            "fusion_strategy",
            "journal_role",
        ]
    ].to_string(index=False)
)

print(f"\nInput audit: {INPUT_AUDIT_CSV}")
print(f"Hybrid paired input manifest: {HYBRID_PAIRED_INPUT_MANIFEST_CSV}")
print(f"Hybrid experiment plan CSV: {HYBRID_EXPERIMENT_PLAN_CSV}")
print(f"Hybrid experiment plan JSON: {HYBRID_EXPERIMENT_PLAN_JSON}")
print(f"Setup summary JSON: {SETUP_SUMMARY_JSON}")

expected_splits = {"train", "val", "test"}

if not expected_splits.issubset(set(hybrid_manifest["glaucoma_split"].astype(str))):
    raise RuntimeError("Hybrid manifest does not contain train/val/test splits.")

if not patient_overlap_df["n_overlap_patients"].eq(0).all():
    raise RuntimeError("Patient overlap detected across hybrid splits.")

if len(hybrid_manifest) != 745:
    raise RuntimeError(f"Unexpected hybrid manifest rows: {len(hybrid_manifest)} != 745")

print("Status: PASS")

In [ ]:
# Cell 2: Prepare hybrid runtime cache, dataloaders, and model smoke test

import time
import shutil
import timm
from PIL import Image
from tqdm.auto import tqdm

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

HYBRID_INPUT_MANIFEST_CSV = (
    HYBRID_DATA_DIR
    / "notebook11_hybrid_full_image_od25x_paired_input_manifest.csv"
)

HYBRID_RUNTIME_CACHE_ROOT = Path("/content/hygd_hybrid_anatomy_aware_inputs_384")
HYBRID_RUNTIME_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

HYBRID_FULL_CACHE_DIR = HYBRID_RUNTIME_CACHE_ROOT / "full_image"
HYBRID_OD25_CACHE_DIR = HYBRID_RUNTIME_CACHE_ROOT / "od_centered_crop_2p5x"

for split_name in ["train", "val", "test"]:
    (HYBRID_FULL_CACHE_DIR / split_name).mkdir(parents=True, exist_ok=True)
    (HYBRID_OD25_CACHE_DIR / split_name).mkdir(parents=True, exist_ok=True)

HYBRID_RUNTIME_MANIFEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_runtime_input_manifest.csv"
)

HYBRID_RUNTIME_CACHE_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_runtime_cache_summary.csv"
)

HYBRID_RUNTIME_CACHE_ERROR_LOG_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_runtime_cache_error_log.csv"
)

HYBRID_DATALOADER_SMOKE_TEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_dataloader_smoke_test.csv"
)

HYBRID_MODEL_SMOKE_TEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_model_smoke_test.csv"
)

HYBRID_RUNTIME_CONFIG_JSON = (
    RESULTS_11_DIR
    / "notebook11_hybrid_runtime_config.json"
)

LOCAL_HYGD_IMAGE_CACHE = Path("/content/HYGD_local/Images")

HYBRID_INPUT_SIZE = 384
HYBRID_FULL_BRANCH_BACKBONE = "efficientnet_b0"
HYBRID_ANATOMY_BRANCH_BACKBONE = "efficientnet_b0"
HYBRID_PRETRAINED = True

HYBRID_TRAIN_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2
HYBRID_EVAL_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2

NUM_WORKERS = 0
PIN_MEMORY = False

FORCE_REBUILD_HYBRID_CACHE = False

if not HYBRID_INPUT_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing hybrid paired manifest: {HYBRID_INPUT_MANIFEST_CSV}")

hybrid_manifest = pd.read_csv(HYBRID_INPUT_MANIFEST_CSV)

required_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "gon_label",
    "glaucoma_split",
    "full_image_path",
    "od_centered_2p5x_image_path",
    "quality_score",
    "area_cdr_proxy",
    "hybrid_experiment_id",
]

missing_cols = [
    col for col in required_cols
    if col not in hybrid_manifest.columns
]

if missing_cols:
    raise ValueError(f"Missing hybrid manifest columns: {missing_cols}")

hybrid_manifest["sample_id"] = hybrid_manifest["sample_id"].astype(str)
hybrid_manifest["patient_id"] = hybrid_manifest["patient_id"].astype(str)
hybrid_manifest["glaucoma_split"] = hybrid_manifest["glaucoma_split"].astype(str)
hybrid_manifest["full_image_path"] = hybrid_manifest["full_image_path"].astype(str)
hybrid_manifest["od_centered_2p5x_image_path"] = hybrid_manifest["od_centered_2p5x_image_path"].astype(str)
hybrid_manifest["gon_binary"] = pd.to_numeric(hybrid_manifest["gon_binary"], errors="coerce").astype(int)


def candidate_full_image_paths(image_path):
    image_path = str(image_path)
    candidates = [
        str(LOCAL_HYGD_IMAGE_CACHE / Path(image_path).name),
        image_path,
    ]

    unique_candidates = []
    seen = set()

    for candidate in candidates:
        if candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)

    return unique_candidates


def read_rgb_image_robust_any(image_path, n_attempts=3, sleep_sec=0.25, use_local_hygd_candidate=True):
    if use_local_hygd_candidate:
        candidates = candidate_full_image_paths(image_path)
    else:
        candidates = [str(image_path)]

    last_error = ""

    for candidate in candidates:
        for _ in range(n_attempts):
            try:
                image_bgr = cv2.imread(str(candidate), cv2.IMREAD_COLOR)

                if image_bgr is not None:
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    return image_rgb, candidate

                last_error = f"cv2.imread returned None for {candidate}"

            except Exception as exc:
                last_error = repr(exc)

            time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read image from candidates for {image_path}. Last error: {last_error}"
    )


def save_rgb_png_local(image_rgb, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(output_path), image_bgr)

    if not ok:
        raise RuntimeError(f"cv2.imwrite failed for {output_path}")

    return str(output_path)


cache_rows = []
cache_error_rows = []

for _, row in tqdm(
    hybrid_manifest.iterrows(),
    total=len(hybrid_manifest),
    desc="Caching hybrid paired inputs locally",
):
    sample_id = str(row["sample_id"])
    split_name = str(row["glaucoma_split"])

    full_runtime_path = (
        HYBRID_FULL_CACHE_DIR
        / split_name
        / f"{sample_id}_full_image.png"
    )

    od25_runtime_path = (
        HYBRID_OD25_CACHE_DIR
        / split_name
        / f"{sample_id}_od_centered_crop_2p5x.png"
    )

    try:
        if full_runtime_path.exists() and not FORCE_REBUILD_HYBRID_CACHE:
            full_cache_status = "already_cached"
            full_resolved_path = str(full_runtime_path)
        else:
            full_rgb, full_resolved_path = read_rgb_image_robust_any(
                row["full_image_path"],
                use_local_hygd_candidate=True,
            )

            save_rgb_png_local(
                full_rgb,
                full_runtime_path,
            )

            full_cache_status = "cached"

        if od25_runtime_path.exists() and not FORCE_REBUILD_HYBRID_CACHE:
            od25_cache_status = "already_cached"
            od25_resolved_path = str(od25_runtime_path)
        else:
            od25_rgb, od25_resolved_path = read_rgb_image_robust_any(
                row["od_centered_2p5x_image_path"],
                use_local_hygd_candidate=False,
            )

            save_rgb_png_local(
                od25_rgb,
                od25_runtime_path,
            )

            od25_cache_status = "cached"

        cache_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": str(row["patient_id"]),
                "glaucoma_split": split_name,
                "full_source_path": str(row["full_image_path"]),
                "od25_source_path": str(row["od_centered_2p5x_image_path"]),
                "full_resolved_path": str(full_resolved_path),
                "od25_resolved_path": str(od25_resolved_path),
                "runtime_full_image_path": str(full_runtime_path),
                "runtime_od25_image_path": str(od25_runtime_path),
                "full_cache_status": full_cache_status,
                "od25_cache_status": od25_cache_status,
                "runtime_full_exists": full_runtime_path.exists(),
                "runtime_od25_exists": od25_runtime_path.exists(),
            }
        )

    except Exception as exc:
        cache_error_rows.append(
            {
                "sample_id": sample_id,
                "patient_id": str(row.get("patient_id", "")),
                "glaucoma_split": split_name,
                "error": repr(exc),
            }
        )

cache_df = pd.DataFrame(cache_rows)
cache_error_df = pd.DataFrame(cache_error_rows)

cache_df.to_csv(HYBRID_RUNTIME_CACHE_SUMMARY_CSV, index=False)
cache_error_df.to_csv(HYBRID_RUNTIME_CACHE_ERROR_LOG_CSV, index=False)

if len(cache_error_df) > 0:
    print(cache_error_df.head(20).to_string(index=False))
    raise RuntimeError(f"Hybrid runtime cache failed for {len(cache_error_df)} cases.")

hybrid_runtime_manifest = hybrid_manifest.merge(
    cache_df[
        [
            "sample_id",
            "patient_id",
            "glaucoma_split",
            "runtime_full_image_path",
            "runtime_od25_image_path",
            "runtime_full_exists",
            "runtime_od25_exists",
        ]
    ],
    on=["sample_id", "patient_id", "glaucoma_split"],
    how="left",
    validate="one_to_one",
)

if hybrid_runtime_manifest["runtime_full_image_path"].isna().any():
    missing_full = int(hybrid_runtime_manifest["runtime_full_image_path"].isna().sum())
    raise RuntimeError(f"Missing runtime full-image paths: {missing_full}")

if hybrid_runtime_manifest["runtime_od25_image_path"].isna().any():
    missing_od25 = int(hybrid_runtime_manifest["runtime_od25_image_path"].isna().sum())
    raise RuntimeError(f"Missing runtime OD-centered paths: {missing_od25}")

if not hybrid_runtime_manifest["runtime_full_exists"].astype(bool).all():
    missing_full_files = int((~hybrid_runtime_manifest["runtime_full_exists"].astype(bool)).sum())
    raise RuntimeError(f"Missing runtime full-image files: {missing_full_files}")

if not hybrid_runtime_manifest["runtime_od25_exists"].astype(bool).all():
    missing_od25_files = int((~hybrid_runtime_manifest["runtime_od25_exists"].astype(bool)).sum())
    raise RuntimeError(f"Missing runtime OD-centered files: {missing_od25_files}")

hybrid_runtime_manifest.to_csv(HYBRID_RUNTIME_MANIFEST_CSV, index=False)


hybrid_train_transform = transforms.Compose(
    [
        transforms.Resize((HYBRID_INPUT_SIZE, HYBRID_INPUT_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.05,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

hybrid_eval_transform = transforms.Compose(
    [
        transforms.Resize((HYBRID_INPUT_SIZE, HYBRID_INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class HybridAnatomyDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        full_image = Image.open(str(row["runtime_full_image_path"])).convert("RGB")
        od25_image = Image.open(str(row["runtime_od25_image_path"])).convert("RGB")

        if self.transform is not None:
            full_tensor = self.transform(full_image)
            od25_tensor = self.transform(od25_image)
        else:
            to_tensor = transforms.ToTensor()
            full_tensor = to_tensor(full_image)
            od25_tensor = to_tensor(od25_image)

        label = torch.tensor(float(row["gon_binary"]), dtype=torch.float32)

        metadata = {
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "gon_label": str(row["gon_label"]),
            "glaucoma_split": str(row["glaucoma_split"]),
            "hybrid_experiment_id": str(row["hybrid_experiment_id"]),
            "quality_score": float(row["quality_score"]) if pd.notna(row["quality_score"]) else np.nan,
            "area_cdr_proxy": float(row["area_cdr_proxy"]) if pd.notna(row["area_cdr_proxy"]) else np.nan,
            "runtime_full_image_path": str(row["runtime_full_image_path"]),
            "runtime_od25_image_path": str(row["runtime_od25_image_path"]),
        }

        return full_tensor, od25_tensor, label, metadata


def collate_hybrid_batch(batch):
    full_images = torch.stack([item[0] for item in batch], dim=0)
    od25_images = torch.stack([item[1] for item in batch], dim=0)
    labels = torch.stack([item[2] for item in batch], dim=0)

    metadata = {}

    for key in batch[0][3].keys():
        metadata[key] = [item[3][key] for item in batch]

    return full_images, od25_images, labels, metadata


def build_hybrid_weighted_sampler(dataframe):
    labels = dataframe["gon_binary"].astype(int).values

    class_counts = {
        class_value: int((labels == class_value).sum())
        for class_value in [0, 1]
    }

    if class_counts[0] == 0 or class_counts[1] == 0:
        return None, class_counts

    class_weights = {
        class_value: 1.0 / class_counts[class_value]
        for class_value in [0, 1]
    }

    sample_weights = np.array(
        [
            class_weights[int(label)]
            for label in labels
        ],
        dtype=np.float64,
    )

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

    return sampler, class_counts


def get_hybrid_split_df(split_name):
    split_df = hybrid_runtime_manifest[
        hybrid_runtime_manifest["glaucoma_split"] == split_name
    ].copy().reset_index(drop=True)

    if len(split_df) == 0:
        raise RuntimeError(f"Empty hybrid split dataframe: {split_name}")

    return split_df


def make_hybrid_dataloaders():
    train_df = get_hybrid_split_df("train")
    val_df = get_hybrid_split_df("val")
    test_df = get_hybrid_split_df("test")

    train_dataset = HybridAnatomyDataset(
        dataframe=train_df,
        transform=hybrid_train_transform,
    )

    val_dataset = HybridAnatomyDataset(
        dataframe=val_df,
        transform=hybrid_eval_transform,
    )

    test_dataset = HybridAnatomyDataset(
        dataframe=test_df,
        transform=hybrid_eval_transform,
    )

    train_sampler, train_class_counts = build_hybrid_weighted_sampler(train_df)

    train_loader = DataLoader(
        train_dataset,
        batch_size=HYBRID_TRAIN_BATCH_SIZE,
        sampler=train_sampler,
        shuffle=(train_sampler is None),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_hybrid_batch,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=HYBRID_EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_hybrid_batch,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=HYBRID_EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_hybrid_batch,
    )

    loaders = {
        "train": train_loader,
        "val": val_loader,
        "test": test_loader,
    }

    split_dfs = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    return loaders, split_dfs, train_class_counts


class HybridEfficientNetB0Classifier(nn.Module):
    def __init__(
        self,
        full_backbone_name=HYBRID_FULL_BRANCH_BACKBONE,
        anatomy_backbone_name=HYBRID_ANATOMY_BRANCH_BACKBONE,
        pretrained=HYBRID_PRETRAINED,
        dropout=0.35,
        hidden_dim=512,
    ):
        super().__init__()

        self.full_branch = timm.create_model(
            full_backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        self.anatomy_branch = timm.create_model(
            anatomy_backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        full_dim = int(self.full_branch.num_features)
        anatomy_dim = int(self.anatomy_branch.num_features)
        fusion_dim = full_dim + anatomy_dim

        self.full_dim = full_dim
        self.anatomy_dim = anatomy_dim
        self.fusion_dim = fusion_dim

        self.fusion_head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, full_image, anatomy_image, return_features=False):
        full_features = self.full_branch(full_image)
        anatomy_features = self.anatomy_branch(anatomy_image)

        fused_features = torch.cat(
            [
                full_features,
                anatomy_features,
            ],
            dim=1,
        )

        logits = self.fusion_head(fused_features).squeeze(1)

        if return_features:
            return {
                "logits": logits,
                "full_features": full_features,
                "anatomy_features": anatomy_features,
                "fused_features": fused_features,
            }

        return logits


def create_hybrid_classifier(pretrained=HYBRID_PRETRAINED):
    model = HybridEfficientNetB0Classifier(
        full_backbone_name=HYBRID_FULL_BRANCH_BACKBONE,
        anatomy_backbone_name=HYBRID_ANATOMY_BRANCH_BACKBONE,
        pretrained=pretrained,
    )

    return model


def compute_hybrid_class_loss_weights(train_df):
    n_negative = int((train_df["gon_binary"].astype(int) == 0).sum())
    n_positive = int((train_df["gon_binary"].astype(int) == 1).sum())
    n_total = n_negative + n_positive

    if n_negative == 0 or n_positive == 0:
        raise RuntimeError("Cannot compute class weights with an empty class.")

    negative_weight = n_total / (2.0 * n_negative)
    positive_weight = n_total / (2.0 * n_positive)

    return {
        "n_negative": n_negative,
        "n_positive": n_positive,
        "negative_weight": float(negative_weight),
        "positive_weight": float(positive_weight),
    }


def compute_hybrid_weighted_bce_loss(logits, labels, negative_weight, positive_weight):
    labels = labels.float()
    logits = logits.squeeze()

    if logits.ndim == 0:
        logits = logits.unsqueeze(0)

    raw_loss = F.binary_cross_entropy_with_logits(
        logits,
        labels,
        reduction="none",
    )

    weights = torch.where(
        labels >= 0.5,
        torch.tensor(float(positive_weight), device=labels.device),
        torch.tensor(float(negative_weight), device=labels.device),
    )

    return (raw_loss * weights).mean()


loaders, split_dfs, train_class_counts = make_hybrid_dataloaders()

dataloader_smoke_rows = []

for split_name in ["train", "val", "test"]:
    loader = loaders[split_name]
    split_df = split_dfs[split_name]

    full_images, od25_images, labels, metadata = next(iter(loader))

    dataloader_smoke_rows.append(
        {
            "split": split_name,
            "n_images": int(len(split_df)),
            "n_patients": int(split_df["patient_id"].nunique()),
            "n_positive": int(split_df["gon_binary"].sum()),
            "n_negative": int(len(split_df) - split_df["gon_binary"].sum()),
            "full_batch_shape": tuple(full_images.shape),
            "od25_batch_shape": tuple(od25_images.shape),
            "label_shape": tuple(labels.shape),
            "label_min": float(labels.min().item()),
            "label_max": float(labels.max().item()),
            "first_sample_id": metadata["sample_id"][0],
            "first_full_runtime_path": metadata["runtime_full_image_path"][0],
            "first_od25_runtime_path": metadata["runtime_od25_image_path"][0],
        }
    )

dataloader_smoke_df = pd.DataFrame(dataloader_smoke_rows)
dataloader_smoke_df.to_csv(HYBRID_DATALOADER_SMOKE_TEST_CSV, index=False)

class_weight_info = compute_hybrid_class_loss_weights(split_dfs["train"])

model = create_hybrid_classifier(pretrained=HYBRID_PRETRAINED).to(DEVICE)
model.train()

full_images, od25_images, labels, metadata = next(iter(loaders["train"]))

full_images = full_images.to(DEVICE)
od25_images = od25_images.to(DEVICE)
labels = labels.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

optimizer.zero_grad(set_to_none=True)

with torch.amp.autocast(
    device_type="cuda",
    enabled=(DEVICE.type == "cuda"),
):
    outputs = model(
        full_images,
        od25_images,
        return_features=True,
    )

    logits = outputs["logits"]

    loss = compute_hybrid_weighted_bce_loss(
        logits=logits,
        labels=labels,
        negative_weight=class_weight_info["negative_weight"],
        positive_weight=class_weight_info["positive_weight"],
    )

loss.backward()
optimizer.step()

trainable_parameters = int(
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

model_smoke_df = pd.DataFrame(
    [
        {
            "hybrid_experiment_id": "hybrid_full_image_plus_od_centered_2p5x_effb0",
            "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
            "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
            "input_size": HYBRID_INPUT_SIZE,
            "full_batch_shape": tuple(full_images.shape),
            "od25_batch_shape": tuple(od25_images.shape),
            "logits_shape": tuple(logits.shape),
            "full_feature_shape": tuple(outputs["full_features"].shape),
            "anatomy_feature_shape": tuple(outputs["anatomy_features"].shape),
            "fused_feature_shape": tuple(outputs["fused_features"].shape),
            "full_feature_dim": int(model.full_dim),
            "anatomy_feature_dim": int(model.anatomy_dim),
            "fusion_dim": int(model.fusion_dim),
            "loss": float(loss.detach().cpu().item()),
            "trainable_parameters": trainable_parameters,
            "n_train_negative": class_weight_info["n_negative"],
            "n_train_positive": class_weight_info["n_positive"],
            "negative_loss_weight": class_weight_info["negative_weight"],
            "positive_loss_weight": class_weight_info["positive_weight"],
            "status": "pass",
        }
    ]
)

model_smoke_df.to_csv(HYBRID_MODEL_SMOKE_TEST_CSV, index=False)

runtime_config = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "cell": "02_prepare_hybrid_runtime_dataloaders_and_model_smoke_test",
    "status": "PASS",
    "device": str(DEVICE),
    "hybrid_runtime_cache_root": str(HYBRID_RUNTIME_CACHE_ROOT),
    "hybrid_runtime_manifest_csv": str(HYBRID_RUNTIME_MANIFEST_CSV),
    "hybrid_train_batch_size": int(HYBRID_TRAIN_BATCH_SIZE),
    "hybrid_eval_batch_size": int(HYBRID_EVAL_BATCH_SIZE),
    "num_workers": int(NUM_WORKERS),
    "pin_memory": bool(PIN_MEMORY),
    "input_size": int(HYBRID_INPUT_SIZE),
    "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
    "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
    "pretrained": bool(HYBRID_PRETRAINED),
    "fusion_strategy": "late_feature_concatenation",
    "class_weight_info": class_weight_info,
    "dataloader_smoke_test_csv": str(HYBRID_DATALOADER_SMOKE_TEST_CSV),
    "model_smoke_test_csv": str(HYBRID_MODEL_SMOKE_TEST_CSV),
    "runtime_cache_summary_csv": str(HYBRID_RUNTIME_CACHE_SUMMARY_CSV),
    "runtime_cache_error_log_csv": str(HYBRID_RUNTIME_CACHE_ERROR_LOG_CSV),
}

with open(HYBRID_RUNTIME_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

del model, optimizer, full_images, od25_images, labels, logits, loss, outputs

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"Device: {DEVICE}")
print(f"Hybrid runtime manifest rows: {len(hybrid_runtime_manifest)}")
print(f"Train batch size: {HYBRID_TRAIN_BATCH_SIZE}")
print(f"Eval batch size: {HYBRID_EVAL_BATCH_SIZE}")
print(f"Num workers: {NUM_WORKERS}")
print(f"Full branch backbone: {HYBRID_FULL_BRANCH_BACKBONE}")
print(f"Anatomy branch backbone: {HYBRID_ANATOMY_BRANCH_BACKBONE}")

print("\nHybrid dataloader smoke test:")
print(dataloader_smoke_df.to_string(index=False))

print("\nHybrid model smoke test:")
print(model_smoke_df.to_string(index=False))

print(f"\nRuntime manifest: {HYBRID_RUNTIME_MANIFEST_CSV}")
print(f"Runtime cache summary: {HYBRID_RUNTIME_CACHE_SUMMARY_CSV}")
print(f"Runtime cache error log: {HYBRID_RUNTIME_CACHE_ERROR_LOG_CSV}")
print(f"Dataloader smoke test CSV: {HYBRID_DATALOADER_SMOKE_TEST_CSV}")
print(f"Model smoke test CSV: {HYBRID_MODEL_SMOKE_TEST_CSV}")
print(f"Runtime config JSON: {HYBRID_RUNTIME_CONFIG_JSON}")

if len(dataloader_smoke_df) != 3:
    raise RuntimeError("Unexpected number of dataloader smoke-test rows.")

if len(model_smoke_df) != 1:
    raise RuntimeError("Unexpected number of model smoke-test rows.")

if not np.isfinite(model_smoke_df["loss"]).all():
    raise RuntimeError("Non-finite hybrid model smoke-test loss found.")

print("Status: PASS")

In [ ]:
# Cell 3: Train hybrid anatomy-aware classifier with validation-only selection

import time
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)
from tqdm.auto import tqdm

HYBRID_EXPERIMENT_ID = "hybrid_full_image_plus_od_centered_2p5x_effb0"

HYBRID_TRAINING_ROOT = RESULTS_11_DIR / "hybrid_training_runs"
HYBRID_TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

HYBRID_CHECKPOINT_ROOT = HYBRID_MODELS_DIR / "training_runs"
HYBRID_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

HYBRID_RUN_DIR = HYBRID_TRAINING_ROOT / HYBRID_EXPERIMENT_ID
HYBRID_RUN_DIR.mkdir(parents=True, exist_ok=True)

HYBRID_CHECKPOINT_DIR = HYBRID_CHECKPOINT_ROOT / HYBRID_EXPERIMENT_ID / f"{HYBRID_EXPERIMENT_ID}_seed{SEED}"
HYBRID_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = HYBRID_CHECKPOINT_DIR / "best_checkpoint.pt"
LAST_CHECKPOINT_PATH = HYBRID_CHECKPOINT_DIR / "last_checkpoint.pt"

HYBRID_TRAINING_LOG_CSV = (
    HYBRID_RUN_DIR
    / f"notebook11_training_log_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TRAINING_RESULT_CSV = (
    HYBRID_RUN_DIR
    / f"notebook11_training_result_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_VAL_IMAGE_PREDICTIONS_CSV = (
    HYBRID_RUN_DIR
    / f"notebook11_val_image_predictions_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_VAL_PATIENT_PREDICTIONS_CSV = (
    HYBRID_RUN_DIR
    / f"notebook11_val_patient_predictions_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TRAINING_CONFIG_JSON = (
    RESULTS_11_DIR
    / "notebook11_hybrid_training_config.json"
)

HYBRID_TRAINING_PROGRESS_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_training_progress.csv"
)

HYBRID_VALIDATION_SELECTION_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

RUN_HYBRID_TRAINING_ON_CUDA_ONLY = True

HYBRID_MAX_EPOCHS = 80
HYBRID_EARLY_STOPPING_PATIENCE = 15
HYBRID_LEARNING_RATE = 5e-5
HYBRID_WEIGHT_DECAY = 1e-4
HYBRID_GRAD_CLIP_NORM = 5.0
HYBRID_AMP_ENABLED = DEVICE.type == "cuda"

HYBRID_PRIMARY_SELECTION_METRIC = "patient_val_auroc"
HYBRID_SECONDARY_SELECTION_METRIC = "patient_val_brier_score"

required_runtime_objects = [
    "make_hybrid_dataloaders",
    "create_hybrid_classifier",
    "compute_hybrid_class_loss_weights",
    "compute_hybrid_weighted_bce_loss",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Missing hybrid runtime functions from Cell 2. "
        f"Run Cell 2 first. Missing: {missing_runtime_objects}"
    )

if DEVICE.type != "cuda" and RUN_HYBRID_TRAINING_ON_CUDA_ONLY:
    print(f"Device: {DEVICE}")
    print("Hybrid training skipped because CUDA is not available.")
    print("Set RUN_HYBRID_TRAINING_ON_CUDA_ONLY = False only for intentional CPU execution.")
    print("Status: PASS")

else:
    def compute_binary_metrics_from_probs_hybrid(y_true, y_prob, threshold=0.5):
        y_true = np.asarray(y_true, dtype=int)
        y_prob = np.asarray(y_prob, dtype=float)
        threshold = float(threshold)

        valid = np.isfinite(y_prob)
        y_true = y_true[valid]
        y_prob = y_prob[valid]

        if len(y_true) == 0:
            raise RuntimeError("No valid samples for metric computation.")

        y_pred = (y_prob >= threshold).astype(int)

        if len(np.unique(y_true)) == 2:
            auroc = float(roc_auc_score(y_true, y_prob))
            auprc = float(average_precision_score(y_true, y_prob))
            logloss = float(
                log_loss(
                    y_true,
                    np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                    labels=[0, 1],
                )
            )
        else:
            auroc = np.nan
            auprc = np.nan
            logloss = np.nan

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        return {
            "auroc": auroc,
            "auprc": auprc,
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "sensitivity": float(sensitivity),
            "specificity": float(specificity),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "brier_score": float(brier_score_loss(y_true, y_prob)),
            "log_loss": logloss,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


    def find_best_threshold_by_balanced_accuracy_hybrid(y_true, y_prob):
        y_true = np.asarray(y_true, dtype=int)
        y_prob = np.asarray(y_prob, dtype=float)

        thresholds = np.round(np.linspace(0.01, 0.99, 99), 4)

        best_threshold = 0.5
        best_balanced_accuracy = -np.inf

        for threshold in thresholds:
            y_pred = (y_prob >= threshold).astype(int)
            score = balanced_accuracy_score(y_true, y_pred)

            if score > best_balanced_accuracy:
                best_balanced_accuracy = float(score)
                best_threshold = float(threshold)

        return best_threshold, best_balanced_accuracy


    def collect_hybrid_predictions(model, loader, split_name):
        model.eval()

        prediction_rows = []

        with torch.no_grad():
            for full_images, od25_images, labels, metadata in tqdm(
                loader,
                desc=f"Collecting {split_name} predictions",
                leave=False,
            ):
                full_images = full_images.to(DEVICE, non_blocking=True)
                od25_images = od25_images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=HYBRID_AMP_ENABLED,
                ):
                    logits = model(full_images, od25_images)
                    probs = torch.sigmoid(logits)

                logits_np = logits.detach().cpu().numpy()
                probs_np = probs.detach().cpu().numpy()
                labels_np = labels.detach().cpu().numpy()

                for idx in range(len(labels_np)):
                    prediction_rows.append(
                        {
                            "sample_id": str(metadata["sample_id"][idx]),
                            "patient_id": str(metadata["patient_id"][idx]),
                            "gon_label": str(metadata["gon_label"][idx]),
                            "glaucoma_split": str(metadata["glaucoma_split"][idx]),
                            "hybrid_experiment_id": str(metadata["hybrid_experiment_id"][idx]),
                            "quality_score": float(metadata["quality_score"][idx]),
                            "area_cdr_proxy": float(metadata["area_cdr_proxy"][idx]),
                            "runtime_full_image_path": str(metadata["runtime_full_image_path"][idx]),
                            "runtime_od25_image_path": str(metadata["runtime_od25_image_path"][idx]),
                            "y_true": int(labels_np[idx]),
                            "logit": float(logits_np[idx]),
                            "y_prob": float(probs_np[idx]),
                        }
                    )

        return pd.DataFrame(prediction_rows)


    def aggregate_hybrid_patient_predictions(image_predictions, aggregation_method="mean"):
        df = image_predictions.copy()

        if aggregation_method == "mean":
            patient_df = (
                df
                .groupby("patient_id")
                .agg(
                    n_images=("sample_id", "nunique"),
                    y_true=("y_true", "max"),
                    y_prob=("y_prob", "mean"),
                    y_prob_mean=("y_prob", "mean"),
                    y_prob_max=("y_prob", "max"),
                    y_prob_median=("y_prob", "median"),
                    quality_score_mean=("quality_score", "mean"),
                    area_cdr_proxy_mean=("area_cdr_proxy", "mean"),
                    hybrid_experiment_id=("hybrid_experiment_id", "first"),
                )
                .reset_index()
            )

        elif aggregation_method == "quality_weighted_mean":
            weighted_rows = []

            for patient_id, patient_block in df.groupby("patient_id"):
                weights = patient_block["quality_score"].astype(float).values
                probs = patient_block["y_prob"].astype(float).values

                if np.isfinite(weights).all() and float(weights.sum()) > 0:
                    weighted_prob = float(np.average(probs, weights=weights))
                else:
                    weighted_prob = float(np.mean(probs))

                weighted_rows.append(
                    {
                        "patient_id": str(patient_id),
                        "n_images": int(patient_block["sample_id"].nunique()),
                        "y_true": int(patient_block["y_true"].max()),
                        "y_prob": weighted_prob,
                        "y_prob_mean": float(patient_block["y_prob"].mean()),
                        "y_prob_max": float(patient_block["y_prob"].max()),
                        "y_prob_median": float(patient_block["y_prob"].median()),
                        "quality_score_mean": float(patient_block["quality_score"].mean()),
                        "area_cdr_proxy_mean": float(patient_block["area_cdr_proxy"].mean()),
                        "hybrid_experiment_id": str(patient_block["hybrid_experiment_id"].iloc[0]),
                    }
                )

            patient_df = pd.DataFrame(weighted_rows)

        else:
            raise ValueError(f"Unknown aggregation method: {aggregation_method}")

        patient_df["aggregation_method"] = aggregation_method

        return patient_df


    def evaluate_hybrid_prediction_tables(image_predictions, patient_predictions):
        image_threshold, image_threshold_bacc = find_best_threshold_by_balanced_accuracy_hybrid(
            image_predictions["y_true"].values,
            image_predictions["y_prob"].values,
        )

        patient_threshold, patient_threshold_bacc = find_best_threshold_by_balanced_accuracy_hybrid(
            patient_predictions["y_true"].values,
            patient_predictions["y_prob"].values,
        )

        image_metrics = compute_binary_metrics_from_probs_hybrid(
            image_predictions["y_true"].values,
            image_predictions["y_prob"].values,
            threshold=image_threshold,
        )

        patient_metrics = compute_binary_metrics_from_probs_hybrid(
            patient_predictions["y_true"].values,
            patient_predictions["y_prob"].values,
            threshold=patient_threshold,
        )

        image_metrics = {
            f"image_val_{key}": value
            for key, value in image_metrics.items()
        }

        patient_metrics = {
            f"patient_val_{key}": value
            for key, value in patient_metrics.items()
        }

        combined_metrics = {
            **image_metrics,
            **patient_metrics,
            "image_val_threshold": float(image_threshold),
            "image_val_threshold_balanced_accuracy": float(image_threshold_bacc),
            "patient_val_threshold": float(patient_threshold),
            "patient_val_threshold_balanced_accuracy": float(patient_threshold_bacc),
        }

        return combined_metrics


    def should_update_hybrid_best(current_metrics, best_metrics):
        if best_metrics is None:
            return True

        current_primary = float(current_metrics[HYBRID_PRIMARY_SELECTION_METRIC])
        best_primary = float(best_metrics[HYBRID_PRIMARY_SELECTION_METRIC])

        if current_primary > best_primary + 1e-8:
            return True

        if abs(current_primary - best_primary) <= 1e-8:
            current_brier = float(current_metrics[HYBRID_SECONDARY_SELECTION_METRIC])
            best_brier = float(best_metrics[HYBRID_SECONDARY_SELECTION_METRIC])

            if current_brier < best_brier - 1e-8:
                return True

            if abs(current_brier - best_brier) <= 1e-8:
                current_bacc = float(current_metrics["patient_val_balanced_accuracy"])
                best_bacc = float(best_metrics["patient_val_balanced_accuracy"])

                if current_bacc > best_bacc + 1e-8:
                    return True

        return False


    def save_hybrid_checkpoint(
        checkpoint_path,
        model,
        optimizer,
        scheduler,
        epoch,
        best_epoch,
        best_metrics,
        patience_counter,
        training_history,
    ):
        checkpoint_payload = {
            "epoch": int(epoch),
            "best_epoch": int(best_epoch),
            "best_metrics": best_metrics,
            "patience_counter": int(patience_counter),
            "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
            "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
            "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
            "input_size": int(HYBRID_INPUT_SIZE),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "training_history": training_history,
            "seed": int(SEED),
        }

        torch.save(checkpoint_payload, checkpoint_path)


    def load_hybrid_checkpoint_if_available(checkpoint_path, model, optimizer, scheduler):
        checkpoint_path = Path(checkpoint_path)

        if not checkpoint_path.exists():
            return {
                "resume": False,
                "start_epoch": 1,
                "best_epoch": 0,
                "best_metrics": None,
                "patience_counter": 0,
                "training_history": [],
            }

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        return {
            "resume": True,
            "start_epoch": int(checkpoint["epoch"]) + 1,
            "best_epoch": int(checkpoint.get("best_epoch", 0)),
            "best_metrics": checkpoint.get("best_metrics", None),
            "patience_counter": int(checkpoint.get("patience_counter", 0)),
            "training_history": checkpoint.get("training_history", []),
        }


    completion_ready = (
        HYBRID_TRAINING_RESULT_CSV.exists()
        and BEST_CHECKPOINT_PATH.exists()
        and HYBRID_VAL_IMAGE_PREDICTIONS_CSV.exists()
        and HYBRID_VAL_PATIENT_PREDICTIONS_CSV.exists()
    )

    if completion_ready:
        result_df = pd.read_csv(HYBRID_TRAINING_RESULT_CSV)

        progress_df = pd.DataFrame(
            [
                {
                    "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
                    "completed": True,
                    "skipped_existing": True,
                    "result_csv": str(HYBRID_TRAINING_RESULT_CSV),
                    "best_checkpoint": str(BEST_CHECKPOINT_PATH),
                    "best_patient_val_auroc": float(result_df["best_patient_val_auroc"].iloc[0]),
                    "best_patient_val_brier_score": float(result_df["best_patient_val_brier_score"].iloc[0]),
                    "best_patient_val_balanced_accuracy": float(result_df["best_patient_val_balanced_accuracy"].iloc[0]),
                    "best_epoch": int(result_df["best_epoch_one_indexed"].iloc[0]),
                    "epochs_ran": int(result_df["epochs_ran"].iloc[0]),
                    "elapsed_min": float(result_df["elapsed_min"].iloc[0]),
                }
            ]
        )

        progress_df.to_csv(HYBRID_TRAINING_PROGRESS_CSV, index=False)

        print("Hybrid training already completed. Existing result loaded:")
        print(progress_df.to_string(index=False))
        print("Status: PASS")

    else:
        loaders, split_dfs, train_class_counts = make_hybrid_dataloaders()
        class_weight_info = compute_hybrid_class_loss_weights(split_dfs["train"])

        model = create_hybrid_classifier(pretrained=HYBRID_PRETRAINED).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=HYBRID_LEARNING_RATE,
            weight_decay=HYBRID_WEIGHT_DECAY,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=HYBRID_MAX_EPOCHS,
        )

        try:
            scaler = torch.amp.GradScaler(
                "cuda",
                enabled=HYBRID_AMP_ENABLED,
            )
        except TypeError:
            scaler = torch.cuda.amp.GradScaler(
                enabled=HYBRID_AMP_ENABLED,
            )

        resume_state = load_hybrid_checkpoint_if_available(
            checkpoint_path=LAST_CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
        )

        start_epoch = resume_state["start_epoch"]
        best_epoch = resume_state["best_epoch"]
        best_metrics = resume_state["best_metrics"]
        patience_counter = resume_state["patience_counter"]
        training_history = resume_state["training_history"]

        if HYBRID_TRAINING_LOG_CSV.exists() and len(training_history) == 0:
            training_history = pd.read_csv(HYBRID_TRAINING_LOG_CSV).to_dict(orient="records")

        training_config = {
            "notebook": "11_hybrid_anatomy_aware_classifier_training",
            "cell": "03_train_hybrid_anatomy_aware_classifier",
            "status": "configured",
            "device": str(DEVICE),
            "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
            "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
            "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
            "fusion_strategy": "late_feature_concatenation",
            "input_size": int(HYBRID_INPUT_SIZE),
            "pretrained": bool(HYBRID_PRETRAINED),
            "max_epochs": int(HYBRID_MAX_EPOCHS),
            "early_stopping_patience": int(HYBRID_EARLY_STOPPING_PATIENCE),
            "learning_rate": float(HYBRID_LEARNING_RATE),
            "weight_decay": float(HYBRID_WEIGHT_DECAY),
            "grad_clip_norm": float(HYBRID_GRAD_CLIP_NORM),
            "amp_enabled": bool(HYBRID_AMP_ENABLED),
            "train_batch_size": int(HYBRID_TRAIN_BATCH_SIZE),
            "eval_batch_size": int(HYBRID_EVAL_BATCH_SIZE),
            "primary_selection_metric": HYBRID_PRIMARY_SELECTION_METRIC,
            "secondary_selection_metric": HYBRID_SECONDARY_SELECTION_METRIC,
            "class_weight_info": class_weight_info,
            "train_class_counts": train_class_counts,
            "best_checkpoint_path": str(BEST_CHECKPOINT_PATH),
            "last_checkpoint_path": str(LAST_CHECKPOINT_PATH),
            "resumed_from_checkpoint": bool(resume_state["resume"]),
            "start_epoch": int(start_epoch),
        }

        with open(HYBRID_TRAINING_CONFIG_JSON, "w", encoding="utf-8") as f:
            json.dump(training_config, f, indent=2)

        print("Hybrid anatomy-aware training started")
        print(f"Experiment: {HYBRID_EXPERIMENT_ID}")
        print(f"Device: {DEVICE}")
        print(f"Start epoch: {start_epoch}/{HYBRID_MAX_EPOCHS}")
        print(f"Train images: {len(split_dfs['train'])}")
        print(f"Val images: {len(split_dfs['val'])}")
        print(f"Train batch size: {HYBRID_TRAIN_BATCH_SIZE}")
        print(f"Class weights: negative={class_weight_info['negative_weight']:.6f}, positive={class_weight_info['positive_weight']:.6f}")
        print(f"Primary selection metric: {HYBRID_PRIMARY_SELECTION_METRIC}")
        print(f"Secondary selection metric: {HYBRID_SECONDARY_SELECTION_METRIC}")

        start_time = time.time()

        for epoch in range(start_epoch, HYBRID_MAX_EPOCHS + 1):
            model.train()

            running_loss = 0.0
            n_train_samples = 0

            train_iterator = tqdm(
                loaders["train"],
                desc=f"Hybrid epoch {epoch:02d}",
                leave=False,
            )

            for full_images, od25_images, labels, metadata in train_iterator:
                full_images = full_images.to(DEVICE, non_blocking=True)
                od25_images = od25_images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=HYBRID_AMP_ENABLED,
                ):
                    logits = model(full_images, od25_images)

                    loss = compute_hybrid_weighted_bce_loss(
                        logits=logits,
                        labels=labels,
                        negative_weight=class_weight_info["negative_weight"],
                        positive_weight=class_weight_info["positive_weight"],
                    )

                scaler.scale(loss).backward()

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=HYBRID_GRAD_CLIP_NORM,
                )

                scaler.step(optimizer)
                scaler.update()

                batch_size = int(labels.shape[0])
                running_loss += float(loss.detach().cpu().item()) * batch_size
                n_train_samples += batch_size

                train_iterator.set_postfix(
                    loss=f"{running_loss / max(n_train_samples, 1):.4f}"
                )

            scheduler.step()

            train_loss = running_loss / max(n_train_samples, 1)

            val_image_predictions = collect_hybrid_predictions(
                model=model,
                loader=loaders["val"],
                split_name="hybrid val",
            )

            val_patient_predictions = aggregate_hybrid_patient_predictions(
                val_image_predictions,
                aggregation_method="mean",
            )

            val_metrics = evaluate_hybrid_prediction_tables(
                image_predictions=val_image_predictions,
                patient_predictions=val_patient_predictions,
            )

            improved = should_update_hybrid_best(
                current_metrics=val_metrics,
                best_metrics=best_metrics,
            )

            if improved:
                best_metrics = val_metrics.copy()
                best_epoch = int(epoch)
                patience_counter = 0

                save_hybrid_checkpoint(
                    checkpoint_path=BEST_CHECKPOINT_PATH,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    best_epoch=best_epoch,
                    best_metrics=best_metrics,
                    patience_counter=patience_counter,
                    training_history=training_history,
                )

                val_image_predictions.to_csv(HYBRID_VAL_IMAGE_PREDICTIONS_CSV, index=False)
                val_patient_predictions.to_csv(HYBRID_VAL_PATIENT_PREDICTIONS_CSV, index=False)

            else:
                patience_counter += 1

            epoch_record = {
                "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
                "epoch": int(epoch),
                "train_loss": float(train_loss),
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
                "improved": bool(improved),
                "best_epoch": int(best_epoch),
                "patience_counter": int(patience_counter),
                **val_metrics,
            }

            training_history.append(epoch_record)
            pd.DataFrame(training_history).to_csv(HYBRID_TRAINING_LOG_CSV, index=False)

            save_hybrid_checkpoint(
                checkpoint_path=LAST_CHECKPOINT_PATH,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                best_epoch=best_epoch,
                best_metrics=best_metrics,
                patience_counter=patience_counter,
                training_history=training_history,
            )

            print(
                f"hybrid | epoch={epoch:02d} | "
                f"train_loss={train_loss:.6f} | "
                f"patient_val_auroc={val_metrics['patient_val_auroc']:.6f} | "
                f"patient_val_brier={val_metrics['patient_val_brier_score']:.6f} | "
                f"patient_val_bacc={val_metrics['patient_val_balanced_accuracy']:.6f} | "
                f"best_epoch={best_epoch} | improved={improved}"
            )

            if patience_counter >= HYBRID_EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered for hybrid model at epoch {epoch}.")
                break

        elapsed_min = round((time.time() - start_time) / 60.0, 3)

        if not BEST_CHECKPOINT_PATH.exists():
            raise RuntimeError("No best checkpoint was saved for the hybrid model.")

        best_checkpoint = torch.load(
            BEST_CHECKPOINT_PATH,
            map_location=DEVICE,
            weights_only=False,
        )

        final_best_metrics = best_checkpoint["best_metrics"]

        trainable_parameters = int(
            sum(p.numel() for p in model.parameters() if p.requires_grad)
        )

        result_row = {
            "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
            "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
            "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
            "fusion_strategy": "late_feature_concatenation",
            "input_size": int(HYBRID_INPUT_SIZE),
            "best_epoch_one_indexed": int(best_checkpoint["best_epoch"]),
            "epochs_ran": int(max([row["epoch"] for row in training_history])),
            "elapsed_min": float(elapsed_min),
            "trainable_parameters": trainable_parameters,
            "best_checkpoint_path": str(BEST_CHECKPOINT_PATH),
            "last_checkpoint_path": str(LAST_CHECKPOINT_PATH),
            "training_log_csv": str(HYBRID_TRAINING_LOG_CSV),
            "val_image_predictions_csv": str(HYBRID_VAL_IMAGE_PREDICTIONS_CSV),
            "val_patient_predictions_csv": str(HYBRID_VAL_PATIENT_PREDICTIONS_CSV),
            "best_image_val_auroc": float(final_best_metrics["image_val_auroc"]),
            "best_image_val_auprc": float(final_best_metrics["image_val_auprc"]),
            "best_image_val_balanced_accuracy": float(final_best_metrics["image_val_balanced_accuracy"]),
            "best_image_val_brier_score": float(final_best_metrics["image_val_brier_score"]),
            "best_image_val_threshold": float(final_best_metrics["image_val_threshold"]),
            "best_patient_val_auroc": float(final_best_metrics["patient_val_auroc"]),
            "best_patient_val_auprc": float(final_best_metrics["patient_val_auprc"]),
            "best_patient_val_balanced_accuracy": float(final_best_metrics["patient_val_balanced_accuracy"]),
            "best_patient_val_brier_score": float(final_best_metrics["patient_val_brier_score"]),
            "best_patient_val_threshold": float(final_best_metrics["patient_val_threshold"]),
            "best_patient_val_sensitivity": float(final_best_metrics["patient_val_sensitivity"]),
            "best_patient_val_specificity": float(final_best_metrics["patient_val_specificity"]),
        }

        pd.DataFrame([result_row]).to_csv(HYBRID_TRAINING_RESULT_CSV, index=False)

        progress_df = pd.DataFrame(
            [
                {
                    "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
                    "completed": True,
                    "skipped_existing": False,
                    "result_csv": str(HYBRID_TRAINING_RESULT_CSV),
                    "best_checkpoint": str(BEST_CHECKPOINT_PATH),
                    "best_patient_val_auroc": float(result_row["best_patient_val_auroc"]),
                    "best_patient_val_brier_score": float(result_row["best_patient_val_brier_score"]),
                    "best_patient_val_balanced_accuracy": float(result_row["best_patient_val_balanced_accuracy"]),
                    "best_epoch": int(result_row["best_epoch_one_indexed"]),
                    "epochs_ran": int(result_row["epochs_ran"]),
                    "elapsed_min": float(result_row["elapsed_min"]),
                }
            ]
        )

        progress_df.to_csv(HYBRID_TRAINING_PROGRESS_CSV, index=False)

        validation_selected_payload = {
            "selection_stage": "validation_only_hybrid_anatomy_aware_model_selection",
            "selected_hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
            "selected_full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
            "selected_anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
            "selected_fusion_strategy": "late_feature_concatenation",
            "selected_best_checkpoint": str(BEST_CHECKPOINT_PATH),
            "selected_best_epoch_one_indexed": int(result_row["best_epoch_one_indexed"]),
            "selected_best_patient_val_auroc": float(result_row["best_patient_val_auroc"]),
            "selected_best_patient_val_auprc": float(result_row["best_patient_val_auprc"]),
            "selected_best_patient_val_balanced_accuracy": float(result_row["best_patient_val_balanced_accuracy"]),
            "selected_best_patient_val_brier_score": float(result_row["best_patient_val_brier_score"]),
            "selected_best_patient_val_threshold": float(result_row["best_patient_val_threshold"]),
            "test_set_policy": (
                "The locked test set has not been used for hybrid model selection. "
                "The selected hybrid checkpoint will be evaluated on the locked test set in the next cell."
            ),
            "training_result_csv": str(HYBRID_TRAINING_RESULT_CSV),
            "training_progress_csv": str(HYBRID_TRAINING_PROGRESS_CSV),
        }

        with open(HYBRID_VALIDATION_SELECTION_JSON, "w", encoding="utf-8") as f:
            json.dump(validation_selected_payload, f, indent=2)

        print("\nHybrid training result:")
        print(pd.DataFrame([result_row]).to_string(index=False))

        print("\nHybrid validation-selected model:")
        print(json.dumps(validation_selected_payload, indent=2))

        print(f"\nTraining log CSV: {HYBRID_TRAINING_LOG_CSV}")
        print(f"Training result CSV: {HYBRID_TRAINING_RESULT_CSV}")
        print(f"Validation image predictions CSV: {HYBRID_VAL_IMAGE_PREDICTIONS_CSV}")
        print(f"Validation patient predictions CSV: {HYBRID_VAL_PATIENT_PREDICTIONS_CSV}")
        print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")
        print(f"Last checkpoint: {LAST_CHECKPOINT_PATH}")
        print(f"Training progress CSV: {HYBRID_TRAINING_PROGRESS_CSV}")
        print(f"Hybrid validation selection JSON: {HYBRID_VALIDATION_SELECTION_JSON}")

        required_outputs = [
            HYBRID_TRAINING_LOG_CSV,
            HYBRID_TRAINING_RESULT_CSV,
            HYBRID_VAL_IMAGE_PREDICTIONS_CSV,
            HYBRID_VAL_PATIENT_PREDICTIONS_CSV,
            BEST_CHECKPOINT_PATH,
            LAST_CHECKPOINT_PATH,
            HYBRID_TRAINING_PROGRESS_CSV,
            HYBRID_VALIDATION_SELECTION_JSON,
            HYBRID_TRAINING_CONFIG_JSON,
        ]

        missing_outputs = [
            str(path)
            for path in required_outputs
            if not Path(path).exists()
        ]

        if missing_outputs:
            raise RuntimeError(f"Missing Cell 3 outputs: {missing_outputs}")

        if not np.isfinite(result_row["best_patient_val_auroc"]):
            raise RuntimeError("Hybrid validation AUROC is non-finite.")

        del model, optimizer, scheduler, scaler

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        print("Status: PASS")

In [ ]:
# Cell 4: Locked-test evaluation for the hybrid anatomy-aware classifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)

HYBRID_EXPERIMENT_ID = "hybrid_full_image_plus_od_centered_2p5x_effb0"

HYBRID_TEST_EVAL_ROOT = RESULTS_11_DIR / "hybrid_locked_test_evaluation"
HYBRID_TEST_EVAL_ROOT.mkdir(parents=True, exist_ok=True)

HYBRID_VALIDATION_SELECTION_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

HYBRID_TRAINING_RESULT_CSV = (
    RESULTS_11_DIR
    / "hybrid_training_runs"
    / HYBRID_EXPERIMENT_ID
    / f"notebook11_training_result_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TEST_IMAGE_PREDICTIONS_CSV = (
    HYBRID_TEST_EVAL_ROOT
    / f"notebook11_test_image_predictions_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TEST_PATIENT_PREDICTIONS_CSV = (
    HYBRID_TEST_EVAL_ROOT
    / f"notebook11_test_patient_predictions_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TEST_IMAGE_METRICS_CSV = (
    HYBRID_TEST_EVAL_ROOT
    / "notebook11_hybrid_test_image_level_metrics.csv"
)

HYBRID_TEST_PATIENT_METRICS_CSV = (
    HYBRID_TEST_EVAL_ROOT
    / "notebook11_hybrid_test_patient_level_metrics.csv"
)

HYBRID_TEST_EVALUATION_SUMMARY_JSON = (
    HYBRID_TEST_EVAL_ROOT
    / "notebook11_hybrid_locked_test_evaluation_summary.json"
)

RUN_HYBRID_TEST_EVALUATION_ON_CUDA_ONLY = True
HYBRID_TEST_AGGREGATION_METHOD = "mean"

required_inputs = [
    HYBRID_VALIDATION_SELECTION_JSON,
    HYBRID_TRAINING_RESULT_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 4 inputs: {missing_inputs}")

required_runtime_objects = [
    "make_hybrid_dataloaders",
    "create_hybrid_classifier",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Missing hybrid runtime functions. Run Cell 2 first. "
        f"Missing: {missing_runtime_objects}"
    )

if DEVICE.type != "cuda" and RUN_HYBRID_TEST_EVALUATION_ON_CUDA_ONLY:
    print(f"Device: {DEVICE}")
    print("Hybrid locked-test evaluation skipped because CUDA is not available.")
    print("Set RUN_HYBRID_TEST_EVALUATION_ON_CUDA_ONLY = False only for intentional CPU execution.")
    print("Status: PASS")

else:
    with open(HYBRID_VALIDATION_SELECTION_JSON, "r", encoding="utf-8") as f:
        validation_selection = json.load(f)

    training_result = pd.read_csv(HYBRID_TRAINING_RESULT_CSV)

    best_checkpoint_path = Path(validation_selection["selected_best_checkpoint"])

    if not best_checkpoint_path.exists():
        raise FileNotFoundError(f"Missing hybrid best checkpoint: {best_checkpoint_path}")

    image_threshold = float(training_result["best_image_val_threshold"].iloc[0])
    patient_threshold = float(training_result["best_patient_val_threshold"].iloc[0])

    def compute_binary_metrics_from_probs_cell4(y_true, y_prob, threshold=0.5):
        y_true = np.asarray(y_true, dtype=int)
        y_prob = np.asarray(y_prob, dtype=float)
        threshold = float(threshold)

        valid = np.isfinite(y_prob)
        y_true = y_true[valid]
        y_prob = y_prob[valid]

        if len(y_true) == 0:
            raise RuntimeError("No valid samples for metric computation.")

        y_pred = (y_prob >= threshold).astype(int)

        if len(np.unique(y_true)) == 2:
            auroc = float(roc_auc_score(y_true, y_prob))
            auprc = float(average_precision_score(y_true, y_prob))
            logloss = float(
                log_loss(
                    y_true,
                    np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                    labels=[0, 1],
                )
            )
        else:
            auroc = np.nan
            auprc = np.nan
            logloss = np.nan

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        return {
            "n_samples": int(len(y_true)),
            "n_positive": int(np.sum(y_true == 1)),
            "n_negative": int(np.sum(y_true == 0)),
            "threshold": float(threshold),
            "auroc": auroc,
            "auprc": auprc,
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "sensitivity": float(sensitivity),
            "specificity": float(specificity),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "brier_score": float(brier_score_loss(y_true, y_prob)),
            "log_loss": logloss,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


    def collect_hybrid_test_predictions(model, loader, split_name):
        model.eval()

        prediction_rows = []

        with torch.no_grad():
            for full_images, od25_images, labels, metadata in tqdm(
                loader,
                desc=f"Collecting {split_name} predictions",
                leave=False,
            ):
                full_images = full_images.to(DEVICE, non_blocking=True)
                od25_images = od25_images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=(DEVICE.type == "cuda"),
                ):
                    logits = model(full_images, od25_images)
                    probs = torch.sigmoid(logits)

                logits_np = logits.detach().cpu().numpy()
                probs_np = probs.detach().cpu().numpy()
                labels_np = labels.detach().cpu().numpy()

                for idx in range(len(labels_np)):
                    prediction_rows.append(
                        {
                            "sample_id": str(metadata["sample_id"][idx]),
                            "patient_id": str(metadata["patient_id"][idx]),
                            "gon_label": str(metadata["gon_label"][idx]),
                            "glaucoma_split": str(metadata["glaucoma_split"][idx]),
                            "hybrid_experiment_id": str(metadata["hybrid_experiment_id"][idx]),
                            "quality_score": float(metadata["quality_score"][idx]),
                            "area_cdr_proxy": float(metadata["area_cdr_proxy"][idx]),
                            "runtime_full_image_path": str(metadata["runtime_full_image_path"][idx]),
                            "runtime_od25_image_path": str(metadata["runtime_od25_image_path"][idx]),
                            "y_true": int(labels_np[idx]),
                            "logit": float(logits_np[idx]),
                            "y_prob": float(probs_np[idx]),
                        }
                    )

        return pd.DataFrame(prediction_rows)


    def aggregate_hybrid_test_patient_predictions(image_predictions, aggregation_method="mean"):
        df = image_predictions.copy()

        if aggregation_method == "mean":
            patient_df = (
                df
                .groupby("patient_id")
                .agg(
                    n_images=("sample_id", "nunique"),
                    y_true=("y_true", "max"),
                    y_prob=("y_prob", "mean"),
                    y_prob_mean=("y_prob", "mean"),
                    y_prob_max=("y_prob", "max"),
                    y_prob_median=("y_prob", "median"),
                    quality_score_mean=("quality_score", "mean"),
                    area_cdr_proxy_mean=("area_cdr_proxy", "mean"),
                    hybrid_experiment_id=("hybrid_experiment_id", "first"),
                )
                .reset_index()
            )

        elif aggregation_method == "quality_weighted_mean":
            weighted_rows = []

            for patient_id, patient_block in df.groupby("patient_id"):
                weights = patient_block["quality_score"].astype(float).values
                probs = patient_block["y_prob"].astype(float).values

                if np.isfinite(weights).all() and float(weights.sum()) > 0:
                    weighted_prob = float(np.average(probs, weights=weights))
                else:
                    weighted_prob = float(np.mean(probs))

                weighted_rows.append(
                    {
                        "patient_id": str(patient_id),
                        "n_images": int(patient_block["sample_id"].nunique()),
                        "y_true": int(patient_block["y_true"].max()),
                        "y_prob": weighted_prob,
                        "y_prob_mean": float(patient_block["y_prob"].mean()),
                        "y_prob_max": float(patient_block["y_prob"].max()),
                        "y_prob_median": float(patient_block["y_prob"].median()),
                        "quality_score_mean": float(patient_block["quality_score"].mean()),
                        "area_cdr_proxy_mean": float(patient_block["area_cdr_proxy"].mean()),
                        "hybrid_experiment_id": str(patient_block["hybrid_experiment_id"].iloc[0]),
                    }
                )

            patient_df = pd.DataFrame(weighted_rows)

        else:
            raise ValueError(f"Unknown aggregation method: {aggregation_method}")

        patient_df["aggregation_method"] = aggregation_method

        return patient_df


    def load_best_hybrid_model(checkpoint_path):
        model = create_hybrid_classifier(pretrained=False).to(DEVICE)

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()

        return model, checkpoint


    print("Hybrid locked-test evaluation")
    print(f"Device: {DEVICE}")
    print(f"Experiment: {HYBRID_EXPERIMENT_ID}")
    print(f"Best checkpoint: {best_checkpoint_path}")
    print(f"Image threshold from validation: {image_threshold:.6f}")
    print(f"Patient threshold from validation: {patient_threshold:.6f}")
    print(f"Aggregation method: {HYBRID_TEST_AGGREGATION_METHOD}")

    if HYBRID_TEST_IMAGE_PREDICTIONS_CSV.exists() and HYBRID_TEST_PATIENT_PREDICTIONS_CSV.exists():
        test_image_predictions = pd.read_csv(HYBRID_TEST_IMAGE_PREDICTIONS_CSV)
        test_patient_predictions = pd.read_csv(HYBRID_TEST_PATIENT_PREDICTIONS_CSV)
        skipped_existing_predictions = True

    else:
        loaders, split_dfs, train_class_counts = make_hybrid_dataloaders()
        test_loader = loaders["test"]

        model, checkpoint = load_best_hybrid_model(best_checkpoint_path)

        test_image_predictions = collect_hybrid_test_predictions(
            model=model,
            loader=test_loader,
            split_name="hybrid locked test",
        )

        test_patient_predictions = aggregate_hybrid_test_patient_predictions(
            test_image_predictions,
            aggregation_method=HYBRID_TEST_AGGREGATION_METHOD,
        )

        test_image_predictions.to_csv(HYBRID_TEST_IMAGE_PREDICTIONS_CSV, index=False)
        test_patient_predictions.to_csv(HYBRID_TEST_PATIENT_PREDICTIONS_CSV, index=False)

        del model, checkpoint

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        skipped_existing_predictions = False

    test_image_predictions["hybrid_experiment_id"] = HYBRID_EXPERIMENT_ID
    test_image_predictions["image_threshold_from_validation"] = image_threshold
    test_image_predictions["patient_threshold_from_validation"] = patient_threshold
    test_image_predictions["y_pred_image_threshold"] = (
        test_image_predictions["y_prob"].astype(float) >= image_threshold
    ).astype(int)
    test_image_predictions["correct_image_threshold"] = (
        test_image_predictions["y_pred_image_threshold"].astype(int)
        == test_image_predictions["y_true"].astype(int)
    )

    test_patient_predictions["hybrid_experiment_id"] = HYBRID_EXPERIMENT_ID
    test_patient_predictions["image_threshold_from_validation"] = image_threshold
    test_patient_predictions["patient_threshold_from_validation"] = patient_threshold
    test_patient_predictions["y_pred_patient_threshold"] = (
        test_patient_predictions["y_prob"].astype(float) >= patient_threshold
    ).astype(int)
    test_patient_predictions["correct_patient_threshold"] = (
        test_patient_predictions["y_pred_patient_threshold"].astype(int)
        == test_patient_predictions["y_true"].astype(int)
    )

    image_metrics = compute_binary_metrics_from_probs_cell4(
        y_true=test_image_predictions["y_true"].values,
        y_prob=test_image_predictions["y_prob"].values,
        threshold=image_threshold,
    )

    patient_metrics = compute_binary_metrics_from_probs_cell4(
        y_true=test_patient_predictions["y_true"].values,
        y_prob=test_patient_predictions["y_prob"].values,
        threshold=patient_threshold,
    )

    image_metrics_df = pd.DataFrame(
        [
            {
                "level": "image",
                "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
                "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
                "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
                "fusion_strategy": "late_feature_concatenation",
                "best_epoch_one_indexed": int(training_result["best_epoch_one_indexed"].iloc[0]),
                **image_metrics,
            }
        ]
    )

    patient_metrics_df = pd.DataFrame(
        [
            {
                "level": "patient",
                "aggregation_method": HYBRID_TEST_AGGREGATION_METHOD,
                "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
                "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
                "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
                "fusion_strategy": "late_feature_concatenation",
                "best_epoch_one_indexed": int(training_result["best_epoch_one_indexed"].iloc[0]),
                **patient_metrics,
            }
        ]
    )

    test_image_predictions.to_csv(HYBRID_TEST_IMAGE_PREDICTIONS_CSV, index=False)
    test_patient_predictions.to_csv(HYBRID_TEST_PATIENT_PREDICTIONS_CSV, index=False)
    image_metrics_df.to_csv(HYBRID_TEST_IMAGE_METRICS_CSV, index=False)
    patient_metrics_df.to_csv(HYBRID_TEST_PATIENT_METRICS_CSV, index=False)

    summary_payload = {
        "notebook": "11_hybrid_anatomy_aware_classifier_training",
        "cell": "04_locked_test_evaluation_hybrid_anatomy_aware_classifier",
        "status": "PASS",
        "device": str(DEVICE),
        "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
        "full_branch_backbone": HYBRID_FULL_BRANCH_BACKBONE,
        "anatomy_branch_backbone": HYBRID_ANATOMY_BRANCH_BACKBONE,
        "fusion_strategy": "late_feature_concatenation",
        "best_checkpoint": str(best_checkpoint_path),
        "best_epoch_one_indexed": int(training_result["best_epoch_one_indexed"].iloc[0]),
        "skipped_existing_predictions": bool(skipped_existing_predictions),
        "image_threshold_from_validation": float(image_threshold),
        "patient_threshold_from_validation": float(patient_threshold),
        "aggregation_method": HYBRID_TEST_AGGREGATION_METHOD,
        "test_image_auroc": float(image_metrics["auroc"]),
        "test_image_auprc": float(image_metrics["auprc"]),
        "test_image_balanced_accuracy": float(image_metrics["balanced_accuracy"]),
        "test_image_brier_score": float(image_metrics["brier_score"]),
        "test_patient_auroc": float(patient_metrics["auroc"]),
        "test_patient_auprc": float(patient_metrics["auprc"]),
        "test_patient_balanced_accuracy": float(patient_metrics["balanced_accuracy"]),
        "test_patient_brier_score": float(patient_metrics["brier_score"]),
        "test_patient_sensitivity": float(patient_metrics["sensitivity"]),
        "test_patient_specificity": float(patient_metrics["specificity"]),
        "test_set_policy": (
            "The locked test set was used only after validation-only hybrid checkpoint selection. "
            "No threshold, architecture, or checkpoint was selected using the locked test set."
        ),
        "outputs": {
            "test_image_predictions_csv": str(HYBRID_TEST_IMAGE_PREDICTIONS_CSV),
            "test_patient_predictions_csv": str(HYBRID_TEST_PATIENT_PREDICTIONS_CSV),
            "test_image_metrics_csv": str(HYBRID_TEST_IMAGE_METRICS_CSV),
            "test_patient_metrics_csv": str(HYBRID_TEST_PATIENT_METRICS_CSV),
        },
    }

    with open(HYBRID_TEST_EVALUATION_SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary_payload, f, indent=2)

    print("\nHybrid locked-test image-level metrics:")
    print(
        image_metrics_df[
            [
                "hybrid_experiment_id",
                "n_samples",
                "auroc",
                "auprc",
                "balanced_accuracy",
                "sensitivity",
                "specificity",
                "brier_score",
                "threshold",
            ]
        ].to_string(index=False)
    )

    print("\nHybrid locked-test patient-level metrics:")
    print(
        patient_metrics_df[
            [
                "hybrid_experiment_id",
                "n_samples",
                "auroc",
                "auprc",
                "balanced_accuracy",
                "sensitivity",
                "specificity",
                "brier_score",
                "threshold",
            ]
        ].to_string(index=False)
    )

    print(f"\nHybrid test image predictions: {HYBRID_TEST_IMAGE_PREDICTIONS_CSV}")
    print(f"Hybrid test patient predictions: {HYBRID_TEST_PATIENT_PREDICTIONS_CSV}")
    print(f"Hybrid test image metrics: {HYBRID_TEST_IMAGE_METRICS_CSV}")
    print(f"Hybrid test patient metrics: {HYBRID_TEST_PATIENT_METRICS_CSV}")
    print(f"Hybrid test summary JSON: {HYBRID_TEST_EVALUATION_SUMMARY_JSON}")

    required_outputs = [
        HYBRID_TEST_IMAGE_PREDICTIONS_CSV,
        HYBRID_TEST_PATIENT_PREDICTIONS_CSV,
        HYBRID_TEST_IMAGE_METRICS_CSV,
        HYBRID_TEST_PATIENT_METRICS_CSV,
        HYBRID_TEST_EVALUATION_SUMMARY_JSON,
    ]

    missing_outputs = [
        str(path)
        for path in required_outputs
        if not Path(path).exists()
    ]

    if missing_outputs:
        raise RuntimeError(f"Missing Cell 4 outputs: {missing_outputs}")

    if not np.isfinite(patient_metrics_df["auroc"]).all():
        raise RuntimeError("Non-finite hybrid patient-level AUROC found.")

    print("Status: PASS")

In [ ]:
# Cell 5: Bootstrap comparison of hybrid model against full-image, anatomy-constrained, and shortcut-control baselines

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
)
from tqdm.auto import tqdm
import hashlib

HYBRID_EXPERIMENT_ID = "hybrid_full_image_plus_od_centered_2p5x_effb0"
HYBRID_MODEL_ID = "hybrid__full_image_plus_od_centered_2p5x_effb0"

HYBRID_COMPARISON_ROOT = RESULTS_11_DIR / "hybrid_vs_internal_baselines_comparison"
HYBRID_COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

NOTEBOOK10_COMBINED_PATIENT_PREDICTIONS_CSV = (
    RESULTS_10_DIR
    / "anatomy_vs_full_image_comparison"
    / "notebook10_combined_full_image_and_anatomy_patient_predictions.csv"
)

HYBRID_TEST_PATIENT_PREDICTIONS_CSV = (
    RESULTS_11_DIR
    / "hybrid_locked_test_evaluation"
    / f"notebook11_test_patient_predictions_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TEST_PATIENT_METRICS_CSV = (
    RESULTS_11_DIR
    / "hybrid_locked_test_evaluation"
    / "notebook11_hybrid_test_patient_level_metrics.csv"
)

COMBINED_WITH_HYBRID_PATIENT_PREDICTIONS_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_combined_internal_and_hybrid_patient_predictions.csv"
)

HYBRID_BOOTSTRAP_CI_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_combined_internal_and_hybrid_bootstrap_ci.csv"
)

HYBRID_PAIRED_COMPARISON_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_paired_hybrid_vs_internal_baselines.csv"
)

HYBRID_INTERPRETATION_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_hybrid_vs_internal_baselines_interpretation.csv"
)

HYBRID_COMPARISON_SUMMARY_JSON = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_hybrid_vs_internal_baselines_comparison_summary.json"
)

PARTIAL_HYBRID_BOOTSTRAP_CI_CSV = (
    HYBRID_COMPARISON_ROOT
    / "PARTIAL_notebook11_combined_internal_and_hybrid_bootstrap_ci.csv"
)

PARTIAL_HYBRID_PAIRED_COMPARISON_CSV = (
    HYBRID_COMPARISON_ROOT
    / "PARTIAL_notebook11_paired_hybrid_vs_internal_baselines.csv"
)

BOOTSTRAP_N = 5000
BOOTSTRAP_SEED = SEED

COMPARISON_METRICS = [
    "auroc",
    "auprc",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
]

required_inputs = [
    NOTEBOOK10_COMBINED_PATIENT_PREDICTIONS_CSV,
    HYBRID_TEST_PATIENT_PREDICTIONS_CSV,
    HYBRID_TEST_PATIENT_METRICS_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 5 inputs: {missing_inputs}")

baseline_predictions = pd.read_csv(NOTEBOOK10_COMBINED_PATIENT_PREDICTIONS_CSV)
hybrid_predictions = pd.read_csv(HYBRID_TEST_PATIENT_PREDICTIONS_CSV)
hybrid_metrics = pd.read_csv(HYBRID_TEST_PATIENT_METRICS_CSV)

baseline_predictions["patient_id"] = baseline_predictions["patient_id"].astype(str)
hybrid_predictions["patient_id"] = hybrid_predictions["patient_id"].astype(str)

required_baseline_cols = [
    "model_id",
    "model_source",
    "model_name",
    "experiment_id",
    "patient_id",
    "y_true",
    "y_prob",
    "threshold",
    "n_images",
]

missing_baseline_cols = [
    col for col in required_baseline_cols
    if col not in baseline_predictions.columns
]

if missing_baseline_cols:
    raise ValueError(f"Missing baseline prediction columns: {missing_baseline_cols}")

required_hybrid_cols = [
    "patient_id",
    "n_images",
    "y_true",
    "y_prob",
    "patient_threshold_from_validation",
]

missing_hybrid_cols = [
    col for col in required_hybrid_cols
    if col not in hybrid_predictions.columns
]

if missing_hybrid_cols:
    raise ValueError(f"Missing hybrid prediction columns: {missing_hybrid_cols}")

hybrid_rows = []

for _, row in hybrid_predictions.iterrows():
    hybrid_rows.append(
        {
            "model_id": HYBRID_MODEL_ID,
            "model_source": "hybrid_anatomy_aware",
            "model_name": "HybridEfficientNetB0_FullImagePlusOD2p5x",
            "experiment_id": HYBRID_EXPERIMENT_ID,
            "patient_id": str(row["patient_id"]),
            "y_true": int(row["y_true"]),
            "y_prob": float(row["y_prob"]),
            "threshold": float(row["patient_threshold_from_validation"]),
            "n_images": int(row["n_images"]),
        }
    )

hybrid_standardized = pd.DataFrame(hybrid_rows)

combined_predictions = pd.concat(
    [
        baseline_predictions[required_baseline_cols],
        hybrid_standardized,
    ],
    axis=0,
    ignore_index=True,
)

model_ids = sorted(combined_predictions["model_id"].astype(str).unique().tolist())

expected_model_ids = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
    HYBRID_MODEL_ID,
]

missing_expected_models = [
    model_id
    for model_id in expected_model_ids
    if model_id not in model_ids
]

if missing_expected_models:
    raise RuntimeError(f"Missing expected models in combined predictions: {missing_expected_models}")

patient_counts = (
    combined_predictions
    .groupby("model_id")
    .agg(
        n_patients=("patient_id", "nunique"),
        n_positive=("y_true", "sum"),
    )
    .reset_index()
)

if int(patient_counts["n_patients"].min()) != int(patient_counts["n_patients"].max()):
    raise RuntimeError("Not all models contain the same number of patients.")

combined_predictions.to_csv(COMBINED_WITH_HYBRID_PATIENT_PREDICTIONS_CSV, index=False)


def stable_seed(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % modulo


def compute_metric_bundle(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    threshold = float(threshold)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return {
            metric: np.nan
            for metric in COMPARISON_METRICS
        }

    y_pred = (y_prob >= threshold).astype(int)

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
    else:
        auroc = np.nan
        auprc = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    if np.isfinite(sensitivity) and np.isfinite(specificity):
        balanced_accuracy = 0.5 * (sensitivity + specificity)
    else:
        balanced_accuracy = np.nan

    return {
        "auroc": auroc,
        "auprc": auprc,
        "balanced_accuracy": float(balanced_accuracy),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
    }


def bootstrap_ci_for_model(model_df, model_id, n_bootstrap, progress_bar=None):
    y_true = model_df["y_true"].astype(int).values
    y_prob = model_df["y_prob"].astype(float).values
    threshold = float(model_df["threshold"].iloc[0])

    observed = compute_metric_bundle(
        y_true=y_true,
        y_prob=y_prob,
        threshold=threshold,
    )

    rng = np.random.default_rng(
        stable_seed("hybrid_ci", model_id, BOOTSTRAP_SEED)
    )

    n = len(model_df)

    boot_values = {
        metric: []
        for metric in COMPARISON_METRICS
    }

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        metrics = compute_metric_bundle(
            y_true=y_true[idx],
            y_prob=y_prob[idx],
            threshold=threshold,
        )

        for metric in COMPARISON_METRICS:
            value = metrics[metric]

            if np.isfinite(value):
                boot_values[metric].append(float(value))

        if progress_bar is not None:
            progress_bar.update(1)

    rows = []

    for metric in COMPARISON_METRICS:
        values = np.asarray(boot_values[metric], dtype=float)

        if len(values) == 0:
            ci_low = np.nan
            ci_high = np.nan
            n_valid = 0
        else:
            ci_low, ci_high = np.percentile(values, [2.5, 97.5])
            n_valid = len(values)

        rows.append(
            {
                "model_id": model_id,
                "model_source": str(model_df["model_source"].iloc[0]),
                "model_name": str(model_df["model_name"].iloc[0]),
                "experiment_id": str(model_df["experiment_id"].iloc[0]),
                "metric": metric,
                "observed": float(observed[metric]),
                "ci95_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
                "ci95_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
                "n_valid_bootstrap": int(n_valid),
                "n_patients": int(len(model_df)),
                "threshold": threshold,
            }
        )

    return rows


def bootstrap_paired_delta(reference_df, comparator_df, reference_id, comparator_id, n_bootstrap, progress_bar=None):
    merged = reference_df[
        [
            "patient_id",
            "y_true",
            "y_prob",
            "threshold",
        ]
    ].rename(
        columns={
            "y_prob": "reference_y_prob",
            "threshold": "reference_threshold",
        }
    ).merge(
        comparator_df[
            [
                "patient_id",
                "y_true",
                "y_prob",
                "threshold",
            ]
        ].rename(
            columns={
                "y_prob": "comparator_y_prob",
                "threshold": "comparator_threshold",
            }
        ),
        on=["patient_id", "y_true"],
        how="inner",
        validate="one_to_one",
    )

    if len(merged) == 0:
        raise RuntimeError(f"No paired patients for {comparator_id} vs {reference_id}")

    y_true = merged["y_true"].astype(int).values
    reference_prob = merged["reference_y_prob"].astype(float).values
    comparator_prob = merged["comparator_y_prob"].astype(float).values
    reference_threshold = float(merged["reference_threshold"].iloc[0])
    comparator_threshold = float(merged["comparator_threshold"].iloc[0])

    reference_observed = compute_metric_bundle(
        y_true=y_true,
        y_prob=reference_prob,
        threshold=reference_threshold,
    )

    comparator_observed = compute_metric_bundle(
        y_true=y_true,
        y_prob=comparator_prob,
        threshold=comparator_threshold,
    )

    rng = np.random.default_rng(
        stable_seed("hybrid_paired", reference_id, comparator_id, BOOTSTRAP_SEED)
    )

    n = len(merged)

    boot_deltas = {
        metric: []
        for metric in COMPARISON_METRICS
    }

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        reference_metrics = compute_metric_bundle(
            y_true=y_true[idx],
            y_prob=reference_prob[idx],
            threshold=reference_threshold,
        )

        comparator_metrics = compute_metric_bundle(
            y_true=y_true[idx],
            y_prob=comparator_prob[idx],
            threshold=comparator_threshold,
        )

        for metric in COMPARISON_METRICS:
            delta = comparator_metrics[metric] - reference_metrics[metric]

            if np.isfinite(delta):
                boot_deltas[metric].append(float(delta))

        if progress_bar is not None:
            progress_bar.update(1)

    rows = []

    for metric in COMPARISON_METRICS:
        values = np.asarray(boot_deltas[metric], dtype=float)
        observed_delta = comparator_observed[metric] - reference_observed[metric]

        if len(values) == 0:
            ci_low = np.nan
            ci_high = np.nan
            p_two_sided = np.nan
            n_valid = 0
        else:
            ci_low, ci_high = np.percentile(values, [2.5, 97.5])

            p_lower_or_equal_zero = np.mean(values <= 0)
            p_greater_or_equal_zero = np.mean(values >= 0)

            p_two_sided = min(
                1.0,
                2.0 * min(p_lower_or_equal_zero, p_greater_or_equal_zero),
            )

            n_valid = len(values)

        rows.append(
            {
                "reference_model_id": reference_id,
                "comparator_model_id": comparator_id,
                "metric": metric,
                "reference_observed": float(reference_observed[metric]),
                "comparator_observed": float(comparator_observed[metric]),
                "delta_comparator_minus_reference": float(observed_delta),
                "delta_ci95_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
                "delta_ci95_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
                "paired_bootstrap_p_two_sided": float(p_two_sided) if np.isfinite(p_two_sided) else np.nan,
                "n_valid_bootstrap": int(n_valid),
                "n_paired_patients": int(len(merged)),
            }
        )

    return rows


def interpret_delta(metric, delta, ci_low, ci_high):
    if metric == "brier_score":
        if delta < 0 and ci_high < 0:
            return "hybrid_significantly_better"
        if delta > 0 and ci_low > 0:
            return "hybrid_significantly_worse"
        return "inconclusive"

    if delta > 0 and ci_low > 0:
        return "hybrid_significantly_better"
    if delta < 0 and ci_high < 0:
        return "hybrid_significantly_worse"

    return "inconclusive"


baseline_model_ids = [
    model_id
    for model_id in expected_model_ids
    if model_id != HYBRID_MODEL_ID
]

comparison_pairs = [
    {
        "reference_model_id": model_id,
        "comparator_model_id": HYBRID_MODEL_ID,
    }
    for model_id in baseline_model_ids
]

total_steps = (
    len(expected_model_ids) * BOOTSTRAP_N
    + len(comparison_pairs) * BOOTSTRAP_N
)

print("Hybrid bootstrap comparison against internal baselines")
print(f"Bootstrap resamples: {BOOTSTRAP_N}")
print(f"Models included: {expected_model_ids}")
print(f"Baseline references: {baseline_model_ids}")
print(f"Hybrid comparator: {HYBRID_MODEL_ID}")
print(f"Comparison pairs: {len(comparison_pairs)}")
print(f"Total progress steps: {total_steps}")

bootstrap_rows = []
paired_rows = []

progress_bar = tqdm(
    total=total_steps,
    desc="Hybrid vs internal baselines bootstrap",
)

for model_id in expected_model_ids:
    model_df = combined_predictions[
        combined_predictions["model_id"] == model_id
    ].copy()

    bootstrap_rows.extend(
        bootstrap_ci_for_model(
            model_df=model_df,
            model_id=model_id,
            n_bootstrap=BOOTSTRAP_N,
            progress_bar=progress_bar,
        )
    )

    pd.DataFrame(bootstrap_rows).to_csv(PARTIAL_HYBRID_BOOTSTRAP_CI_CSV, index=False)

for pair in comparison_pairs:
    reference_id = pair["reference_model_id"]
    comparator_id = pair["comparator_model_id"]

    reference_df = combined_predictions[
        combined_predictions["model_id"] == reference_id
    ].copy()

    comparator_df = combined_predictions[
        combined_predictions["model_id"] == comparator_id
    ].copy()

    paired_rows.extend(
        bootstrap_paired_delta(
            reference_df=reference_df,
            comparator_df=comparator_df,
            reference_id=reference_id,
            comparator_id=comparator_id,
            n_bootstrap=BOOTSTRAP_N,
            progress_bar=progress_bar,
        )
    )

    pd.DataFrame(paired_rows).to_csv(PARTIAL_HYBRID_PAIRED_COMPARISON_CSV, index=False)

progress_bar.close()

bootstrap_ci = pd.DataFrame(bootstrap_rows)
paired_comparison = pd.DataFrame(paired_rows)

bootstrap_ci.to_csv(HYBRID_BOOTSTRAP_CI_CSV, index=False)
paired_comparison.to_csv(HYBRID_PAIRED_COMPARISON_CSV, index=False)

interpretation_rows = []

for _, row in paired_comparison.iterrows():
    interpretation_rows.append(
        {
            "reference_model_id": str(row["reference_model_id"]),
            "comparator_model_id": str(row["comparator_model_id"]),
            "metric": str(row["metric"]),
            "reference_observed": float(row["reference_observed"]),
            "hybrid_observed": float(row["comparator_observed"]),
            "delta_hybrid_minus_reference": float(row["delta_comparator_minus_reference"]),
            "delta_ci95_low": float(row["delta_ci95_low"]) if pd.notna(row["delta_ci95_low"]) else np.nan,
            "delta_ci95_high": float(row["delta_ci95_high"]) if pd.notna(row["delta_ci95_high"]) else np.nan,
            "paired_bootstrap_p_two_sided": float(row["paired_bootstrap_p_two_sided"]) if pd.notna(row["paired_bootstrap_p_two_sided"]) else np.nan,
            "interpretation": interpret_delta(
                metric=str(row["metric"]),
                delta=float(row["delta_comparator_minus_reference"]),
                ci_low=float(row["delta_ci95_low"]) if pd.notna(row["delta_ci95_low"]) else np.nan,
                ci_high=float(row["delta_ci95_high"]) if pd.notna(row["delta_ci95_high"]) else np.nan,
            ),
        }
    )

interpretation_df = pd.DataFrame(interpretation_rows)
interpretation_df.to_csv(HYBRID_INTERPRETATION_CSV, index=False)

primary_reference_ids = [
    "full_image__EfficientNetB0",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__outside_od_only_trained_control",
]

primary_pair_summary = paired_comparison[
    paired_comparison["reference_model_id"].isin(primary_reference_ids)
].copy()

summary_payload = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "cell": "05_bootstrap_comparison_hybrid_vs_internal_baselines",
    "status": "PASS",
    "bootstrap_n": int(BOOTSTRAP_N),
    "n_models": int(len(expected_model_ids)),
    "n_baseline_references": int(len(baseline_model_ids)),
    "n_comparison_pairs": int(len(comparison_pairs)),
    "n_patients_per_model": int(patient_counts["n_patients"].iloc[0]),
    "hybrid_model_id": HYBRID_MODEL_ID,
    "primary_pair_metrics": primary_pair_summary.to_dict(orient="records"),
    "interpretation": (
        "The hybrid model is compared against full-image, anatomy-constrained, and shortcut-control baselines using paired patient-level bootstrap. "
        "The hybrid should be interpreted as a candidate final model only if it preserves internal performance while later showing stronger anatomical plausibility, calibration, and external generalization."
    ),
    "outputs": {
        "combined_patient_predictions_csv": str(COMBINED_WITH_HYBRID_PATIENT_PREDICTIONS_CSV),
        "bootstrap_ci_csv": str(HYBRID_BOOTSTRAP_CI_CSV),
        "paired_comparison_csv": str(HYBRID_PAIRED_COMPARISON_CSV),
        "interpretation_csv": str(HYBRID_INTERPRETATION_CSV),
        "partial_bootstrap_ci_csv": str(PARTIAL_HYBRID_BOOTSTRAP_CI_CSV),
        "partial_paired_comparison_csv": str(PARTIAL_HYBRID_PAIRED_COMPARISON_CSV),
    },
}

with open(HYBRID_COMPARISON_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("\nCombined patient-level bootstrap CI, primary metrics:")
print(
    bootstrap_ci[
        bootstrap_ci["metric"].isin(["auroc", "balanced_accuracy", "brier_score"])
    ][
        [
            "model_id",
            "metric",
            "observed",
            "ci95_low",
            "ci95_high",
            "n_valid_bootstrap",
        ]
    ].sort_values(["model_id", "metric"]).to_string(index=False)
)

print("\nHybrid paired comparison against selected references, primary metrics:")
print(
    paired_comparison[
        paired_comparison["metric"].isin(["auroc", "balanced_accuracy", "brier_score"])
        & paired_comparison["reference_model_id"].isin(primary_reference_ids)
    ][
        [
            "reference_model_id",
            "comparator_model_id",
            "metric",
            "reference_observed",
            "comparator_observed",
            "delta_comparator_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
        ]
    ].sort_values(["reference_model_id", "metric"]).to_string(index=False)
)

print("\nHybrid interpretation flags, primary metrics:")
print(
    interpretation_df[
        interpretation_df["metric"].isin(["auroc", "balanced_accuracy", "brier_score"])
    ][
        [
            "reference_model_id",
            "metric",
            "delta_hybrid_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
            "interpretation",
        ]
    ].sort_values(["reference_model_id", "metric"]).to_string(index=False)
)

print(f"\nCombined predictions CSV: {COMBINED_WITH_HYBRID_PATIENT_PREDICTIONS_CSV}")
print(f"Bootstrap CI CSV: {HYBRID_BOOTSTRAP_CI_CSV}")
print(f"Paired comparison CSV: {HYBRID_PAIRED_COMPARISON_CSV}")
print(f"Interpretation CSV: {HYBRID_INTERPRETATION_CSV}")
print(f"Summary JSON: {HYBRID_COMPARISON_SUMMARY_JSON}")

required_outputs = [
    COMBINED_WITH_HYBRID_PATIENT_PREDICTIONS_CSV,
    HYBRID_BOOTSTRAP_CI_CSV,
    HYBRID_PAIRED_COMPARISON_CSV,
    HYBRID_INTERPRETATION_CSV,
    HYBRID_COMPARISON_SUMMARY_JSON,
    PARTIAL_HYBRID_BOOTSTRAP_CI_CSV,
    PARTIAL_HYBRID_PAIRED_COMPARISON_CSV,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 5 outputs: {missing_outputs}")

if len(bootstrap_ci) != len(expected_model_ids) * len(COMPARISON_METRICS):
    raise RuntimeError("Unexpected number of bootstrap CI rows.")

if len(paired_comparison) != len(comparison_pairs) * len(COMPARISON_METRICS):
    raise RuntimeError("Unexpected number of paired comparison rows.")

if HYBRID_MODEL_ID not in set(bootstrap_ci["model_id"].astype(str)):
    raise RuntimeError("Hybrid model missing from bootstrap CI.")

print("Status: PASS")

In [ ]:
# Cell 6: Generate hybrid versus internal baseline comparison figures

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

HYBRID_MODEL_ID = "hybrid__full_image_plus_od_centered_2p5x_effb0"

HYBRID_COMPARISON_ROOT = RESULTS_11_DIR / "hybrid_vs_internal_baselines_comparison"
FIGURE_ROOT = RESULTS_11_DIR / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

HYBRID_BOOTSTRAP_CI_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_combined_internal_and_hybrid_bootstrap_ci.csv"
)

HYBRID_PAIRED_COMPARISON_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_paired_hybrid_vs_internal_baselines.csv"
)

HYBRID_INTERPRETATION_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_hybrid_vs_internal_baselines_interpretation.csv"
)

MAIN_HYBRID_COMPARISON_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook11_main_hybrid_vs_internal_baselines_comparison.png"
)

MAIN_HYBRID_COMPARISON_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook11_main_hybrid_vs_internal_baselines_comparison.pdf"
)

MAIN_HYBRID_COMPARISON_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook11_main_hybrid_vs_internal_baselines_comparison.svg"
)

HYBRID_PRIMARY_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_primary_hybrid_vs_internal_baselines_summary.csv"
)

HYBRID_PRIMARY_DELTAS_CSV = (
    RESULTS_11_DIR
    / "notebook11_primary_hybrid_deltas_vs_internal_baselines.csv"
)

HYBRID_FIGURE_MANIFEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_comparison_figure_manifest.csv"
)

HYBRID_FIGURE_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_hybrid_comparison_figure_summary.json"
)

required_inputs = [
    HYBRID_BOOTSTRAP_CI_CSV,
    HYBRID_PAIRED_COMPARISON_CSV,
    HYBRID_INTERPRETATION_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 6 inputs: {missing_inputs}")

bootstrap_ci = pd.read_csv(HYBRID_BOOTSTRAP_CI_CSV)
paired_comparison = pd.read_csv(HYBRID_PAIRED_COMPARISON_CSV)
interpretation_df = pd.read_csv(HYBRID_INTERPRETATION_CSV)

MODEL_ORDER = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
    HYBRID_MODEL_ID,
]

MODEL_LABELS = {
    "full_image__EfficientNetB0": "Full image\nEffNet-B0",
    "full_image__MobileNetV3Large": "Full image\nMobileNetV3",
    "full_image__DeiTSmall": "Full image\nDeiT-Small",
    "anatomy__od_centered_crop_2p5x": "OD-centered\n2.5x",
    "anatomy__od_centered_crop_4p0x": "OD-centered\n4.0x",
    "anatomy__od_only_context_suppressed": "OD-only\ntrained",
    "anatomy__outside_od_only_trained_control": "Outside-OD\ncontrol",
    HYBRID_MODEL_ID: "Hybrid\nFull + OD 2.5x",
}

MODEL_GROUPS = {
    "full_image__EfficientNetB0": "Full-image",
    "full_image__MobileNetV3Large": "Full-image",
    "full_image__DeiTSmall": "Full-image",
    "anatomy__od_centered_crop_2p5x": "Anatomy-constrained",
    "anatomy__od_centered_crop_4p0x": "Anatomy-constrained",
    "anatomy__od_only_context_suppressed": "Anatomy stress test",
    "anatomy__outside_od_only_trained_control": "Shortcut-control",
    HYBRID_MODEL_ID: "Hybrid anatomy-aware",
}

MODEL_COLORS = {
    "Full-image": "#4C78A8",
    "Anatomy-constrained": "#54A24B",
    "Anatomy stress test": "#F58518",
    "Shortcut-control": "#B279A2",
    "Hybrid anatomy-aware": "#E45756",
}

REFERENCE_ORDER = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
]

PRIMARY_METRICS = [
    "auroc",
    "balanced_accuracy",
    "brier_score",
]

METRIC_LABELS = {
    "auroc": "Patient AUROC",
    "balanced_accuracy": "Patient balanced accuracy",
    "brier_score": "Patient Brier score",
}

available_models = set(bootstrap_ci["model_id"].astype(str).unique())

missing_models = [
    model_id
    for model_id in MODEL_ORDER
    if model_id not in available_models
]

if missing_models:
    raise RuntimeError(f"Missing expected models from bootstrap CI: {missing_models}")


def get_bootstrap_row(model_id, metric):
    block = bootstrap_ci[
        (bootstrap_ci["model_id"].astype(str) == model_id)
        & (bootstrap_ci["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(f"Bootstrap row not found uniquely: {model_id} | {metric}")

    return block.iloc[0]


def get_paired_row(reference_model_id, metric):
    block = paired_comparison[
        (paired_comparison["reference_model_id"].astype(str) == reference_model_id)
        & (paired_comparison["comparator_model_id"].astype(str) == HYBRID_MODEL_ID)
        & (paired_comparison["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(
            f"Paired row not found uniquely: {reference_model_id} | {HYBRID_MODEL_ID} | {metric}"
        )

    return block.iloc[0]


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )


def style_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid_axis, alpha=0.25, linewidth=0.8)
    ax.tick_params(axis="both", labelsize=9)


summary_rows = []

for model_id in MODEL_ORDER:
    for metric in PRIMARY_METRICS:
        row = get_bootstrap_row(model_id, metric)

        summary_rows.append(
            {
                "model_id": model_id,
                "model_label": MODEL_LABELS[model_id],
                "model_group": MODEL_GROUPS[model_id],
                "metric": metric,
                "metric_label": METRIC_LABELS[metric],
                "observed": float(row["observed"]),
                "ci95_low": float(row["ci95_low"]),
                "ci95_high": float(row["ci95_high"]),
                "n_valid_bootstrap": int(row["n_valid_bootstrap"]),
            }
        )

primary_summary = pd.DataFrame(summary_rows)
primary_summary.to_csv(HYBRID_PRIMARY_SUMMARY_CSV, index=False)

delta_rows = []

for reference_model_id in REFERENCE_ORDER:
    for metric in PRIMARY_METRICS:
        row = get_paired_row(reference_model_id, metric)

        delta_rows.append(
            {
                "reference_model_id": reference_model_id,
                "reference_label": MODEL_LABELS[reference_model_id],
                "reference_group": MODEL_GROUPS[reference_model_id],
                "comparator_model_id": HYBRID_MODEL_ID,
                "metric": metric,
                "metric_label": METRIC_LABELS[metric],
                "reference_observed": float(row["reference_observed"]),
                "hybrid_observed": float(row["comparator_observed"]),
                "delta_hybrid_minus_reference": float(row["delta_comparator_minus_reference"]),
                "delta_ci95_low": float(row["delta_ci95_low"]),
                "delta_ci95_high": float(row["delta_ci95_high"]),
                "paired_bootstrap_p_two_sided": float(row["paired_bootstrap_p_two_sided"]),
            }
        )

primary_deltas = pd.DataFrame(delta_rows)
primary_deltas.to_csv(HYBRID_PRIMARY_DELTAS_CSV, index=False)

fig = plt.figure(figsize=(19, 10.8))

gs = gridspec.GridSpec(
    2,
    3,
    figure=fig,
    height_ratios=[1.0, 1.0],
    width_ratios=[1.0, 1.0, 1.0],
    hspace=0.52,
    wspace=0.36,
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
]

x = np.arange(len(MODEL_ORDER))

bar_colors = [
    MODEL_COLORS[MODEL_GROUPS[model_id]]
    for model_id in MODEL_ORDER
]

for ax_idx, metric in enumerate(PRIMARY_METRICS):
    ax = axes[ax_idx]

    values = []
    lower_errors = []
    upper_errors = []

    for model_id in MODEL_ORDER:
        row = get_bootstrap_row(model_id, metric)

        observed = float(row["observed"])
        ci_low = float(row["ci95_low"])
        ci_high = float(row["ci95_high"])

        values.append(observed)
        lower_errors.append(abs(observed - ci_low))
        upper_errors.append(abs(ci_high - observed))

    ax.bar(
        x,
        values,
        color=bar_colors,
        alpha=0.88,
    )

    ax.errorbar(
        x,
        values,
        yerr=[lower_errors, upper_errors],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
    )

    ax.axvline(2.5, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
    ax.axvline(6.5, color="black", linestyle="--", linewidth=0.8, alpha=0.4)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [MODEL_LABELS[m] for m in MODEL_ORDER],
        rotation=35,
        ha="right",
    )

    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(f"{METRIC_LABELS[metric]} with 95% bootstrap CI")

    if metric in ["auroc", "balanced_accuracy"]:
        ax.set_ylim(0.75, 1.03)
    else:
        max_value = max(values)
        ax.set_ylim(0.0, max(max_value * 1.45, 0.03))

    style_axis(ax, grid_axis="y")
    add_panel_label(ax, chr(ord("A") + ax_idx))

for delta_idx, metric in enumerate(PRIMARY_METRICS):
    ax = axes[3 + delta_idx]

    metric_block = primary_deltas[
        primary_deltas["metric"] == metric
    ].copy()

    metric_block["order"] = metric_block["reference_model_id"].map(
        {model_id: idx for idx, model_id in enumerate(REFERENCE_ORDER)}
    )

    metric_block = metric_block.sort_values("order").reset_index(drop=True)

    y_positions = np.arange(len(metric_block))

    values = metric_block["delta_hybrid_minus_reference"].astype(float).values
    lower_errors = np.abs(values - metric_block["delta_ci95_low"].astype(float).values)
    upper_errors = np.abs(metric_block["delta_ci95_high"].astype(float).values - values)

    colors = [
        MODEL_COLORS[MODEL_GROUPS[model_id]]
        for model_id in metric_block["reference_model_id"].astype(str).tolist()
    ]

    ax.barh(
        y_positions,
        values,
        color=colors,
        alpha=0.88,
    )

    ax.errorbar(
        values,
        y_positions,
        xerr=[lower_errors, upper_errors],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
    )

    ax.axvline(0, color="black", linewidth=1.0)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        [
            MODEL_LABELS[model_id]
            for model_id in metric_block["reference_model_id"].astype(str).tolist()
        ]
    )

    ax.set_xlabel(
        f"Δ {METRIC_LABELS[metric]}\n"
        "(hybrid − reference)"
    )

    ax.set_title("Paired patient-level delta")
    style_axis(ax, grid_axis="x")
    add_panel_label(ax, chr(ord("D") + delta_idx))

legend_handles = []

for group_name, color in MODEL_COLORS.items():
    if group_name in set(MODEL_GROUPS[m] for m in MODEL_ORDER):
        legend_handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="s",
                color="w",
                label=group_name,
                markerfacecolor=color,
                markersize=10,
            )
        )

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=5,
    frameon=False,
    fontsize=10,
)

fig.suptitle(
    "Hybrid anatomy-aware classifier versus full-image, anatomy-constrained and shortcut-control baselines",
    fontsize=15,
    fontweight="bold",
    y=1.055,
)

plt.savefig(MAIN_HYBRID_COMPARISON_FIGURE_PNG, dpi=300, bbox_inches="tight")
plt.savefig(MAIN_HYBRID_COMPARISON_FIGURE_PDF, bbox_inches="tight")
plt.savefig(MAIN_HYBRID_COMPARISON_FIGURE_SVG, bbox_inches="tight")
plt.show()

figure_manifest = pd.DataFrame(
    [
        {
            "figure_name": "main_hybrid_vs_internal_baselines_comparison",
            "format": "png",
            "figure_path": str(MAIN_HYBRID_COMPARISON_FIGURE_PNG),
            "exists": MAIN_HYBRID_COMPARISON_FIGURE_PNG.exists(),
        },
        {
            "figure_name": "main_hybrid_vs_internal_baselines_comparison",
            "format": "pdf",
            "figure_path": str(MAIN_HYBRID_COMPARISON_FIGURE_PDF),
            "exists": MAIN_HYBRID_COMPARISON_FIGURE_PDF.exists(),
        },
        {
            "figure_name": "main_hybrid_vs_internal_baselines_comparison",
            "format": "svg",
            "figure_path": str(MAIN_HYBRID_COMPARISON_FIGURE_SVG),
            "exists": MAIN_HYBRID_COMPARISON_FIGURE_SVG.exists(),
        },
    ]
)

figure_manifest.to_csv(HYBRID_FIGURE_MANIFEST_CSV, index=False)

significant_brier_improvements = primary_deltas[
    (primary_deltas["metric"] == "brier_score")
    & (primary_deltas["delta_hybrid_minus_reference"] < 0)
    & (primary_deltas["delta_ci95_high"] < 0)
].copy()

figure_summary = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "cell": "06_generate_hybrid_vs_internal_baseline_figures",
    "status": "PASS",
    "hybrid_model_id": HYBRID_MODEL_ID,
    "models_included": MODEL_ORDER,
    "reference_models_for_paired_delta": REFERENCE_ORDER,
    "main_figure_png": str(MAIN_HYBRID_COMPARISON_FIGURE_PNG),
    "main_figure_pdf": str(MAIN_HYBRID_COMPARISON_FIGURE_PDF),
    "main_figure_svg": str(MAIN_HYBRID_COMPARISON_FIGURE_SVG),
    "primary_summary_csv": str(HYBRID_PRIMARY_SUMMARY_CSV),
    "primary_deltas_csv": str(HYBRID_PRIMARY_DELTAS_CSV),
    "figure_manifest_csv": str(HYBRID_FIGURE_MANIFEST_CSV),
    "main_interpretation": (
        "The hybrid model preserves patient-level AUROC and balanced accuracy at ceiling on the locked test set. "
        "Its strongest internal advantage is calibration-like probabilistic performance, reflected in lower Brier score "
        "against selected baselines. Anatomical plausibility still requires XAI and ablation analysis before final model freezing."
    ),
    "significant_brier_improvements_vs_references": significant_brier_improvements[
        [
            "reference_model_id",
            "hybrid_observed",
            "reference_observed",
            "delta_hybrid_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
        ]
    ].to_dict(orient="records"),
}

with open(HYBRID_FIGURE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(figure_summary, f, indent=2)

print("Hybrid primary comparison summary:")
print(
    primary_summary[
        [
            "model_id",
            "metric",
            "observed",
            "ci95_low",
            "ci95_high",
            "n_valid_bootstrap",
        ]
    ].to_string(index=False)
)

print("\nHybrid primary deltas:")
print(primary_deltas.to_string(index=False))

print("\nSignificant Brier improvements:")
if len(significant_brier_improvements) == 0:
    print("None")
else:
    print(
        significant_brier_improvements[
            [
                "reference_model_id",
                "reference_observed",
                "hybrid_observed",
                "delta_hybrid_minus_reference",
                "delta_ci95_low",
                "delta_ci95_high",
                "paired_bootstrap_p_two_sided",
            ]
        ].to_string(index=False)
    )

print("\nGenerated figures:")
print(figure_manifest.to_string(index=False))

print(f"\nPrimary summary CSV: {HYBRID_PRIMARY_SUMMARY_CSV}")
print(f"Primary deltas CSV: {HYBRID_PRIMARY_DELTAS_CSV}")
print(f"Figure manifest CSV: {HYBRID_FIGURE_MANIFEST_CSV}")
print(f"Figure summary JSON: {HYBRID_FIGURE_SUMMARY_JSON}")

if not figure_manifest["exists"].astype(bool).all():
    missing_figures = figure_manifest.loc[
        ~figure_manifest["exists"].astype(bool),
        "figure_path",
    ].tolist()
    raise RuntimeError(f"Missing generated figures: {missing_figures}")

if HYBRID_MODEL_ID not in set(primary_summary["model_id"].astype(str)):
    raise RuntimeError("Hybrid model missing from primary summary.")

if len(primary_deltas) != len(REFERENCE_ORDER) * len(PRIMARY_METRICS):
    raise RuntimeError("Unexpected number of hybrid primary delta rows.")

required_outputs = [
    MAIN_HYBRID_COMPARISON_FIGURE_PNG,
    MAIN_HYBRID_COMPARISON_FIGURE_PDF,
    MAIN_HYBRID_COMPARISON_FIGURE_SVG,
    HYBRID_PRIMARY_SUMMARY_CSV,
    HYBRID_PRIMARY_DELTAS_CSV,
    HYBRID_FIGURE_MANIFEST_CSV,
    HYBRID_FIGURE_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 6 outputs: {missing_outputs}")

print("Status: PASS")

In [ ]:
# Cell 7: Hybrid branch contribution analysis on the locked test set

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)
from tqdm.auto import tqdm
import hashlib

HYBRID_EXPERIMENT_ID = "hybrid_full_image_plus_od_centered_2p5x_effb0"
HYBRID_MODEL_ID = "hybrid__full_image_plus_od_centered_2p5x_effb0"

BRANCH_ANALYSIS_ROOT = RESULTS_11_DIR / "hybrid_branch_contribution_analysis"
BRANCH_ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

HYBRID_VALIDATION_SELECTION_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

HYBRID_TRAINING_RESULT_CSV = (
    RESULTS_11_DIR
    / "hybrid_training_runs"
    / HYBRID_EXPERIMENT_ID
    / f"notebook11_training_result_{HYBRID_EXPERIMENT_ID}.csv"
)

BRANCH_IMAGE_PREDICTIONS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_image_predictions.csv"
)

BRANCH_PATIENT_PREDICTIONS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_patient_predictions.csv"
)

BRANCH_IMAGE_METRICS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_image_metrics.csv"
)

BRANCH_PATIENT_METRICS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_patient_metrics.csv"
)

BRANCH_PAIRED_BOOTSTRAP_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_paired_bootstrap.csv"
)

BRANCH_ANALYSIS_SUMMARY_JSON = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_summary.json"
)

PARTIAL_BRANCH_BOOTSTRAP_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "PARTIAL_notebook11_hybrid_branch_contribution_paired_bootstrap.csv"
)

BRANCH_BOOTSTRAP_N = 5000
BRANCH_BOOTSTRAP_SEED = SEED

BRANCH_MODES = [
    {
        "branch_mode": "original_hybrid",
        "description": "Full-image branch and OD-centered branch both active.",
        "neutralize_full_branch": False,
        "neutralize_od_branch": False,
    },
    {
        "branch_mode": "full_image_branch_only",
        "description": "Full-image branch active; OD-centered branch replaced by normalized neutral input.",
        "neutralize_full_branch": False,
        "neutralize_od_branch": True,
    },
    {
        "branch_mode": "od_centered_branch_only",
        "description": "OD-centered branch active; full-image branch replaced by normalized neutral input.",
        "neutralize_full_branch": True,
        "neutralize_od_branch": False,
    },
    {
        "branch_mode": "both_branches_neutralized",
        "description": "Both branches replaced by normalized neutral input as a negative control.",
        "neutralize_full_branch": True,
        "neutralize_od_branch": True,
    },
]

BRANCH_METRICS = [
    "auroc",
    "auprc",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
]

required_inputs = [
    HYBRID_VALIDATION_SELECTION_JSON,
    HYBRID_TRAINING_RESULT_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 7 inputs: {missing_inputs}")

required_runtime_objects = [
    "make_hybrid_dataloaders",
    "create_hybrid_classifier",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Missing hybrid runtime functions. Run Cell 2 first. "
        f"Missing: {missing_runtime_objects}"
    )

with open(HYBRID_VALIDATION_SELECTION_JSON, "r", encoding="utf-8") as f:
    validation_selection = json.load(f)

training_result = pd.read_csv(HYBRID_TRAINING_RESULT_CSV)

best_checkpoint_path = Path(validation_selection["selected_best_checkpoint"])

if not best_checkpoint_path.exists():
    raise FileNotFoundError(f"Missing hybrid best checkpoint: {best_checkpoint_path}")

image_threshold = float(training_result["best_image_val_threshold"].iloc[0])
patient_threshold = float(training_result["best_patient_val_threshold"].iloc[0])


def compute_binary_metrics_branch(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    threshold = float(threshold)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        raise RuntimeError("No valid samples for metric computation.")

    y_pred = (y_prob >= threshold).astype(int)

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
        logloss = float(
            log_loss(
                y_true,
                np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                labels=[0, 1],
            )
        )
    else:
        auroc = np.nan
        auprc = np.nan
        logloss = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "n_samples": int(len(y_true)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
        "threshold": float(threshold),
        "auroc": auroc,
        "auprc": auprc,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "log_loss": logloss,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def aggregate_branch_patient_predictions(image_predictions, aggregation_method="mean"):
    patient_df = (
        image_predictions
        .groupby(["branch_mode", "patient_id"])
        .agg(
            n_images=("sample_id", "nunique"),
            y_true=("y_true", "max"),
            y_prob=("y_prob", "mean"),
            y_prob_mean=("y_prob", "mean"),
            y_prob_max=("y_prob", "max"),
            y_prob_median=("y_prob", "median"),
            quality_score_mean=("quality_score", "mean"),
            area_cdr_proxy_mean=("area_cdr_proxy", "mean"),
            hybrid_experiment_id=("hybrid_experiment_id", "first"),
            branch_description=("branch_description", "first"),
        )
        .reset_index()
    )

    patient_df["aggregation_method"] = aggregation_method

    return patient_df


def load_best_hybrid_model_for_branch_analysis(checkpoint_path):
    model = create_hybrid_classifier(pretrained=False).to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, checkpoint


def collect_branch_predictions(model, loader):
    model.eval()

    rows = []

    with torch.no_grad():
        for full_images, od25_images, labels, metadata in tqdm(
            loader,
            desc="Hybrid branch contribution inference",
            leave=False,
        ):
            full_images = full_images.to(DEVICE, non_blocking=True)
            od25_images = od25_images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            neutral_full = torch.zeros_like(full_images)
            neutral_od25 = torch.zeros_like(od25_images)

            for mode in BRANCH_MODES:
                input_full = neutral_full if mode["neutralize_full_branch"] else full_images
                input_od25 = neutral_od25 if mode["neutralize_od_branch"] else od25_images

                with torch.amp.autocast(
                    device_type="cuda",
                    enabled=(DEVICE.type == "cuda"),
                ):
                    logits = model(input_full, input_od25)
                    probs = torch.sigmoid(logits)

                logits_np = logits.detach().cpu().numpy()
                probs_np = probs.detach().cpu().numpy()
                labels_np = labels.detach().cpu().numpy()

                for idx in range(len(labels_np)):
                    rows.append(
                        {
                            "branch_mode": mode["branch_mode"],
                            "branch_description": mode["description"],
                            "neutralize_full_branch": bool(mode["neutralize_full_branch"]),
                            "neutralize_od_branch": bool(mode["neutralize_od_branch"]),
                            "sample_id": str(metadata["sample_id"][idx]),
                            "patient_id": str(metadata["patient_id"][idx]),
                            "gon_label": str(metadata["gon_label"][idx]),
                            "glaucoma_split": str(metadata["glaucoma_split"][idx]),
                            "hybrid_experiment_id": str(metadata["hybrid_experiment_id"][idx]),
                            "quality_score": float(metadata["quality_score"][idx]),
                            "area_cdr_proxy": float(metadata["area_cdr_proxy"][idx]),
                            "y_true": int(labels_np[idx]),
                            "logit": float(logits_np[idx]),
                            "y_prob": float(probs_np[idx]),
                        }
                    )

    return pd.DataFrame(rows)


def stable_seed_branch(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()

    return int(digest[:8], 16) % modulo


def bootstrap_paired_branch_delta(reference_df, comparator_df, reference_mode, comparator_mode, n_bootstrap):
    merged = reference_df[
        [
            "patient_id",
            "y_true",
            "y_prob",
        ]
    ].rename(
        columns={
            "y_prob": "reference_y_prob",
        }
    ).merge(
        comparator_df[
            [
                "patient_id",
                "y_true",
                "y_prob",
            ]
        ].rename(
            columns={
                "y_prob": "comparator_y_prob",
            }
        ),
        on=["patient_id", "y_true"],
        how="inner",
        validate="one_to_one",
    )

    if len(merged) == 0:
        raise RuntimeError(f"No paired patients for {comparator_mode} vs {reference_mode}")

    y_true = merged["y_true"].astype(int).values
    reference_prob = merged["reference_y_prob"].astype(float).values
    comparator_prob = merged["comparator_y_prob"].astype(float).values

    reference_observed = compute_binary_metrics_branch(
        y_true=y_true,
        y_prob=reference_prob,
        threshold=patient_threshold,
    )

    comparator_observed = compute_binary_metrics_branch(
        y_true=y_true,
        y_prob=comparator_prob,
        threshold=patient_threshold,
    )

    rng = np.random.default_rng(
        stable_seed_branch(
            "branch_paired",
            reference_mode,
            comparator_mode,
            BRANCH_BOOTSTRAP_SEED,
        )
    )

    n = len(merged)

    boot_deltas = {
        metric: []
        for metric in BRANCH_METRICS
    }

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        reference_metrics = compute_binary_metrics_branch(
            y_true=y_true[idx],
            y_prob=reference_prob[idx],
            threshold=patient_threshold,
        )

        comparator_metrics = compute_binary_metrics_branch(
            y_true=y_true[idx],
            y_prob=comparator_prob[idx],
            threshold=patient_threshold,
        )

        for metric in BRANCH_METRICS:
            delta = comparator_metrics[metric] - reference_metrics[metric]

            if np.isfinite(delta):
                boot_deltas[metric].append(float(delta))

    rows = []

    for metric in BRANCH_METRICS:
        values = np.asarray(boot_deltas[metric], dtype=float)
        observed_delta = comparator_observed[metric] - reference_observed[metric]

        if len(values) == 0:
            ci_low = np.nan
            ci_high = np.nan
            p_two_sided = np.nan
            n_valid = 0
        else:
            ci_low, ci_high = np.percentile(values, [2.5, 97.5])

            p_lower_or_equal_zero = np.mean(values <= 0)
            p_greater_or_equal_zero = np.mean(values >= 0)

            p_two_sided = min(
                1.0,
                2.0 * min(p_lower_or_equal_zero, p_greater_or_equal_zero),
            )

            n_valid = len(values)

        rows.append(
            {
                "reference_branch_mode": reference_mode,
                "comparator_branch_mode": comparator_mode,
                "metric": metric,
                "reference_observed": float(reference_observed[metric]),
                "comparator_observed": float(comparator_observed[metric]),
                "delta_comparator_minus_reference": float(observed_delta),
                "delta_ci95_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
                "delta_ci95_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
                "paired_bootstrap_p_two_sided": float(p_two_sided) if np.isfinite(p_two_sided) else np.nan,
                "n_valid_bootstrap": int(n_valid),
                "n_paired_patients": int(len(merged)),
            }
        )

    return rows


print("Hybrid branch contribution analysis")
print(f"Device: {DEVICE}")
print(f"Best checkpoint: {best_checkpoint_path}")
print(f"Image threshold from validation: {image_threshold:.6f}")
print(f"Patient threshold from validation: {patient_threshold:.6f}")
print(f"Branch modes: {[mode['branch_mode'] for mode in BRANCH_MODES]}")

loaders, split_dfs, train_class_counts = make_hybrid_dataloaders()
test_loader = loaders["test"]

model, checkpoint = load_best_hybrid_model_for_branch_analysis(best_checkpoint_path)

branch_image_predictions = collect_branch_predictions(
    model=model,
    loader=test_loader,
)

del model, checkpoint

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

branch_patient_predictions = aggregate_branch_patient_predictions(
    branch_image_predictions,
    aggregation_method="mean",
)

branch_image_predictions["image_threshold_from_validation"] = image_threshold
branch_image_predictions["patient_threshold_from_validation"] = patient_threshold
branch_image_predictions["y_pred_image_threshold"] = (
    branch_image_predictions["y_prob"].astype(float) >= image_threshold
).astype(int)
branch_image_predictions["correct_image_threshold"] = (
    branch_image_predictions["y_pred_image_threshold"].astype(int)
    == branch_image_predictions["y_true"].astype(int)
)

branch_patient_predictions["image_threshold_from_validation"] = image_threshold
branch_patient_predictions["patient_threshold_from_validation"] = patient_threshold
branch_patient_predictions["y_pred_patient_threshold"] = (
    branch_patient_predictions["y_prob"].astype(float) >= patient_threshold
).astype(int)
branch_patient_predictions["correct_patient_threshold"] = (
    branch_patient_predictions["y_pred_patient_threshold"].astype(int)
    == branch_patient_predictions["y_true"].astype(int)
)

image_metric_rows = []
patient_metric_rows = []

for mode in [mode["branch_mode"] for mode in BRANCH_MODES]:
    image_block = branch_image_predictions[
        branch_image_predictions["branch_mode"] == mode
    ].copy()

    patient_block = branch_patient_predictions[
        branch_patient_predictions["branch_mode"] == mode
    ].copy()

    image_metrics = compute_binary_metrics_branch(
        y_true=image_block["y_true"].values,
        y_prob=image_block["y_prob"].values,
        threshold=image_threshold,
    )

    patient_metrics = compute_binary_metrics_branch(
        y_true=patient_block["y_true"].values,
        y_prob=patient_block["y_prob"].values,
        threshold=patient_threshold,
    )

    image_metric_rows.append(
        {
            "level": "image",
            "branch_mode": mode,
            "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
            **image_metrics,
        }
    )

    patient_metric_rows.append(
        {
            "level": "patient",
            "branch_mode": mode,
            "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
            **patient_metrics,
        }
    )

image_metrics_df = pd.DataFrame(image_metric_rows)
patient_metrics_df = pd.DataFrame(patient_metric_rows)

bootstrap_rows = []

reference_mode = "original_hybrid"

reference_df = branch_patient_predictions[
    branch_patient_predictions["branch_mode"] == reference_mode
].copy()

for comparator_mode in [
    "full_image_branch_only",
    "od_centered_branch_only",
    "both_branches_neutralized",
]:
    comparator_df = branch_patient_predictions[
        branch_patient_predictions["branch_mode"] == comparator_mode
    ].copy()

    bootstrap_rows.extend(
        bootstrap_paired_branch_delta(
            reference_df=reference_df,
            comparator_df=comparator_df,
            reference_mode=reference_mode,
            comparator_mode=comparator_mode,
            n_bootstrap=BRANCH_BOOTSTRAP_N,
        )
    )

    pd.DataFrame(bootstrap_rows).to_csv(PARTIAL_BRANCH_BOOTSTRAP_CSV, index=False)

branch_bootstrap_df = pd.DataFrame(bootstrap_rows)

branch_image_predictions.to_csv(BRANCH_IMAGE_PREDICTIONS_CSV, index=False)
branch_patient_predictions.to_csv(BRANCH_PATIENT_PREDICTIONS_CSV, index=False)
image_metrics_df.to_csv(BRANCH_IMAGE_METRICS_CSV, index=False)
patient_metrics_df.to_csv(BRANCH_PATIENT_METRICS_CSV, index=False)
branch_bootstrap_df.to_csv(BRANCH_PAIRED_BOOTSTRAP_CSV, index=False)

summary_payload = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "cell": "07_hybrid_branch_contribution_analysis",
    "status": "PASS",
    "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
    "hybrid_model_id": HYBRID_MODEL_ID,
    "best_checkpoint": str(best_checkpoint_path),
    "branch_modes": BRANCH_MODES,
    "image_threshold_from_validation": float(image_threshold),
    "patient_threshold_from_validation": float(patient_threshold),
    "bootstrap_n": int(BRANCH_BOOTSTRAP_N),
    "patient_level_metrics": patient_metrics_df.to_dict(orient="records"),
    "primary_interpretation": (
        "This analysis tests whether the hybrid model depends on both full-image and OD-centered branches. "
        "Neutralized inputs are zero tensors after ImageNet normalization, corresponding approximately to mean-color inputs."
    ),
    "outputs": {
        "branch_image_predictions_csv": str(BRANCH_IMAGE_PREDICTIONS_CSV),
        "branch_patient_predictions_csv": str(BRANCH_PATIENT_PREDICTIONS_CSV),
        "branch_image_metrics_csv": str(BRANCH_IMAGE_METRICS_CSV),
        "branch_patient_metrics_csv": str(BRANCH_PATIENT_METRICS_CSV),
        "branch_paired_bootstrap_csv": str(BRANCH_PAIRED_BOOTSTRAP_CSV),
        "partial_branch_bootstrap_csv": str(PARTIAL_BRANCH_BOOTSTRAP_CSV),
    },
}

with open(BRANCH_ANALYSIS_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("\nBranch contribution image-level metrics:")
print(
    image_metrics_df[
        [
            "branch_mode",
            "n_samples",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "brier_score",
            "threshold",
        ]
    ].to_string(index=False)
)

print("\nBranch contribution patient-level metrics:")
print(
    patient_metrics_df[
        [
            "branch_mode",
            "n_samples",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "brier_score",
            "threshold",
        ]
    ].to_string(index=False)
)

print("\nPaired bootstrap: branch modes vs original hybrid:")
print(
    branch_bootstrap_df[
        branch_bootstrap_df["metric"].isin(["auroc", "balanced_accuracy", "brier_score"])
    ][
        [
            "reference_branch_mode",
            "comparator_branch_mode",
            "metric",
            "reference_observed",
            "comparator_observed",
            "delta_comparator_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
        ]
    ].to_string(index=False)
)

print(f"\nBranch image predictions CSV: {BRANCH_IMAGE_PREDICTIONS_CSV}")
print(f"Branch patient predictions CSV: {BRANCH_PATIENT_PREDICTIONS_CSV}")
print(f"Branch image metrics CSV: {BRANCH_IMAGE_METRICS_CSV}")
print(f"Branch patient metrics CSV: {BRANCH_PATIENT_METRICS_CSV}")
print(f"Branch paired bootstrap CSV: {BRANCH_PAIRED_BOOTSTRAP_CSV}")
print(f"Branch analysis summary JSON: {BRANCH_ANALYSIS_SUMMARY_JSON}")

required_outputs = [
    BRANCH_IMAGE_PREDICTIONS_CSV,
    BRANCH_PATIENT_PREDICTIONS_CSV,
    BRANCH_IMAGE_METRICS_CSV,
    BRANCH_PATIENT_METRICS_CSV,
    BRANCH_PAIRED_BOOTSTRAP_CSV,
    PARTIAL_BRANCH_BOOTSTRAP_CSV,
    BRANCH_ANALYSIS_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 7 outputs: {missing_outputs}")

if len(patient_metrics_df) != len(BRANCH_MODES):
    raise RuntimeError("Unexpected number of patient-level branch metric rows.")

if not np.isfinite(patient_metrics_df["auroc"]).all():
    raise RuntimeError("Non-finite patient-level branch AUROC found.")

print("Status: PASS")

In [ ]:
# Cell 8: Generate hybrid branch contribution figure

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

BRANCH_ANALYSIS_ROOT = RESULTS_11_DIR / "hybrid_branch_contribution_analysis"
FIGURE_ROOT = RESULTS_11_DIR / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

BRANCH_PATIENT_METRICS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_patient_metrics.csv"
)

BRANCH_IMAGE_METRICS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_image_metrics.csv"
)

BRANCH_PAIRED_BOOTSTRAP_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_paired_bootstrap.csv"
)

BRANCH_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook11_hybrid_branch_contribution_figure.png"
)

BRANCH_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook11_hybrid_branch_contribution_figure.pdf"
)

BRANCH_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook11_hybrid_branch_contribution_figure.svg"
)

BRANCH_FIGURE_MANIFEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_figure_manifest.csv"
)

BRANCH_FIGURE_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_figure_summary.json"
)

BRANCH_PRIMARY_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_primary_summary.csv"
)

required_inputs = [
    BRANCH_PATIENT_METRICS_CSV,
    BRANCH_IMAGE_METRICS_CSV,
    BRANCH_PAIRED_BOOTSTRAP_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 8 inputs: {missing_inputs}")

patient_metrics = pd.read_csv(BRANCH_PATIENT_METRICS_CSV)
image_metrics = pd.read_csv(BRANCH_IMAGE_METRICS_CSV)
branch_bootstrap = pd.read_csv(BRANCH_PAIRED_BOOTSTRAP_CSV)

BRANCH_ORDER = [
    "original_hybrid",
    "full_image_branch_only",
    "od_centered_branch_only",
    "both_branches_neutralized",
]

BRANCH_LABELS = {
    "original_hybrid": "Hybrid\nFull + OD",
    "full_image_branch_only": "Full-image\nbranch only",
    "od_centered_branch_only": "OD-centered\nbranch only",
    "both_branches_neutralized": "Both branches\nneutralized",
}

BRANCH_COLORS = {
    "original_hybrid": "#E45756",
    "full_image_branch_only": "#4C78A8",
    "od_centered_branch_only": "#54A24B",
    "both_branches_neutralized": "#9D9D9D",
}

PRIMARY_METRICS = [
    "auroc",
    "balanced_accuracy",
    "brier_score",
]

METRIC_LABELS = {
    "auroc": "Patient AUROC",
    "balanced_accuracy": "Patient balanced accuracy",
    "brier_score": "Patient Brier score",
}


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )


def style_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid_axis, alpha=0.25, linewidth=0.8)
    ax.tick_params(axis="both", labelsize=9)


def get_metric_value(df, branch_mode, metric):
    block = df[df["branch_mode"].astype(str) == branch_mode]

    if len(block) != 1:
        raise RuntimeError(f"Metric row not found uniquely: {branch_mode}")

    return float(block[metric].iloc[0])


def get_delta_row(comparator_branch_mode, metric):
    block = branch_bootstrap[
        (branch_bootstrap["reference_branch_mode"].astype(str) == "original_hybrid")
        & (branch_bootstrap["comparator_branch_mode"].astype(str) == comparator_branch_mode)
        & (branch_bootstrap["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(
            f"Delta row not found uniquely: original_hybrid vs {comparator_branch_mode} | {metric}"
        )

    return block.iloc[0]


summary_rows = []

for branch_mode in BRANCH_ORDER:
    for metric in PRIMARY_METRICS:
        summary_rows.append(
            {
                "branch_mode": branch_mode,
                "branch_label": BRANCH_LABELS[branch_mode],
                "metric": metric,
                "metric_label": METRIC_LABELS[metric],
                "patient_value": get_metric_value(patient_metrics, branch_mode, metric),
                "image_value": get_metric_value(image_metrics, branch_mode, metric),
            }
        )

primary_summary = pd.DataFrame(summary_rows)
primary_summary.to_csv(BRANCH_PRIMARY_SUMMARY_CSV, index=False)

fig = plt.figure(figsize=(17.5, 9.5))

gs = gridspec.GridSpec(
    2,
    3,
    figure=fig,
    height_ratios=[1.0, 1.0],
    width_ratios=[1.0, 1.0, 1.0],
    hspace=0.46,
    wspace=0.36,
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
]

x = np.arange(len(BRANCH_ORDER))
colors = [BRANCH_COLORS[branch] for branch in BRANCH_ORDER]

for ax_idx, metric in enumerate(PRIMARY_METRICS):
    ax = axes[ax_idx]

    values = [
        get_metric_value(patient_metrics, branch_mode, metric)
        for branch_mode in BRANCH_ORDER
    ]

    ax.bar(
        x,
        values,
        color=colors,
        alpha=0.88,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [BRANCH_LABELS[branch] for branch in BRANCH_ORDER],
        rotation=25,
        ha="right",
    )

    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(f"{METRIC_LABELS[metric]} by active branch mode")

    if metric in ["auroc", "balanced_accuracy"]:
        ax.set_ylim(0.45, 1.04)
    else:
        ax.set_ylim(0.0, max(values) * 1.25)

    style_axis(ax, grid_axis="y")
    add_panel_label(ax, chr(ord("A") + ax_idx))

COMPARATOR_BRANCHES = [
    "full_image_branch_only",
    "od_centered_branch_only",
    "both_branches_neutralized",
]

for delta_idx, metric in enumerate(PRIMARY_METRICS):
    ax = axes[3 + delta_idx]

    y_positions = np.arange(len(COMPARATOR_BRANCHES))

    values = []
    lower_errors = []
    upper_errors = []
    delta_colors = []

    for comparator_branch in COMPARATOR_BRANCHES:
        row = get_delta_row(comparator_branch, metric)

        delta = float(row["delta_comparator_minus_reference"])
        ci_low = float(row["delta_ci95_low"])
        ci_high = float(row["delta_ci95_high"])

        values.append(delta)
        lower_errors.append(abs(delta - ci_low))
        upper_errors.append(abs(ci_high - delta))
        delta_colors.append(BRANCH_COLORS[comparator_branch])

    ax.barh(
        y_positions,
        values,
        color=delta_colors,
        alpha=0.88,
    )

    ax.errorbar(
        values,
        y_positions,
        xerr=[lower_errors, upper_errors],
        fmt="none",
        ecolor="black",
        elinewidth=0.8,
        capsize=2,
    )

    ax.axvline(0, color="black", linewidth=1.0)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        [BRANCH_LABELS[branch] for branch in COMPARATOR_BRANCHES]
    )

    ax.set_xlabel(
        f"Δ {METRIC_LABELS[metric]}\n"
        "(branch mode − original hybrid)"
    )

    ax.set_title("Paired patient-level branch ablation delta")

    style_axis(ax, grid_axis="x")
    add_panel_label(ax, chr(ord("D") + delta_idx))

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        label=BRANCH_LABELS[branch].replace("\n", " "),
        markerfacecolor=BRANCH_COLORS[branch],
        markersize=10,
    )
    for branch in BRANCH_ORDER
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=4,
    frameon=False,
    fontsize=10,
)

fig.suptitle(
    "Hybrid branch contribution analysis under locked-test branch neutralization",
    fontsize=15,
    fontweight="bold",
    y=1.07,
)

plt.savefig(BRANCH_FIGURE_PNG, dpi=300, bbox_inches="tight")
plt.savefig(BRANCH_FIGURE_PDF, bbox_inches="tight")
plt.savefig(BRANCH_FIGURE_SVG, bbox_inches="tight")
plt.show()

figure_manifest = pd.DataFrame(
    [
        {
            "figure_name": "hybrid_branch_contribution_figure",
            "format": "png",
            "figure_path": str(BRANCH_FIGURE_PNG),
            "exists": BRANCH_FIGURE_PNG.exists(),
        },
        {
            "figure_name": "hybrid_branch_contribution_figure",
            "format": "pdf",
            "figure_path": str(BRANCH_FIGURE_PDF),
            "exists": BRANCH_FIGURE_PDF.exists(),
        },
        {
            "figure_name": "hybrid_branch_contribution_figure",
            "format": "svg",
            "figure_path": str(BRANCH_FIGURE_SVG),
            "exists": BRANCH_FIGURE_SVG.exists(),
        },
    ]
)

figure_manifest.to_csv(BRANCH_FIGURE_MANIFEST_CSV, index=False)

branch_interpretation = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "cell": "08_generate_hybrid_branch_contribution_figure",
    "status": "PASS",
    "main_figure_png": str(BRANCH_FIGURE_PNG),
    "main_figure_pdf": str(BRANCH_FIGURE_PDF),
    "main_figure_svg": str(BRANCH_FIGURE_SVG),
    "branch_primary_summary_csv": str(BRANCH_PRIMARY_SUMMARY_CSV),
    "branch_figure_manifest_csv": str(BRANCH_FIGURE_MANIFEST_CSV),
    "primary_interpretation": (
        "The full-image branch carries stronger standalone predictive signal than the OD-centered branch, "
        "but the original hybrid provides the best combined decision. Neutralizing the OD-centered branch significantly worsens Brier score, "
        "while neutralizing the full-image branch produces larger losses in balanced accuracy and Brier score. "
        "This supports complementary branch contribution rather than a purely redundant hybrid."
    ),
    "patient_level_original_hybrid": patient_metrics[
        patient_metrics["branch_mode"] == "original_hybrid"
    ].to_dict(orient="records"),
    "patient_level_full_branch_only": patient_metrics[
        patient_metrics["branch_mode"] == "full_image_branch_only"
    ].to_dict(orient="records"),
    "patient_level_od_branch_only": patient_metrics[
        patient_metrics["branch_mode"] == "od_centered_branch_only"
    ].to_dict(orient="records"),
    "patient_level_both_neutralized": patient_metrics[
        patient_metrics["branch_mode"] == "both_branches_neutralized"
    ].to_dict(orient="records"),
}

with open(BRANCH_FIGURE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(branch_interpretation, f, indent=2)

print("Branch primary summary:")
print(primary_summary.to_string(index=False))

print("\nBranch paired bootstrap primary deltas:")
print(
    branch_bootstrap[
        branch_bootstrap["metric"].isin(PRIMARY_METRICS)
    ][
        [
            "reference_branch_mode",
            "comparator_branch_mode",
            "metric",
            "reference_observed",
            "comparator_observed",
            "delta_comparator_minus_reference",
            "delta_ci95_low",
            "delta_ci95_high",
            "paired_bootstrap_p_two_sided",
        ]
    ].to_string(index=False)
)

print("\nGenerated figures:")
print(figure_manifest.to_string(index=False))

print(f"\nBranch primary summary CSV: {BRANCH_PRIMARY_SUMMARY_CSV}")
print(f"Branch figure manifest CSV: {BRANCH_FIGURE_MANIFEST_CSV}")
print(f"Branch figure summary JSON: {BRANCH_FIGURE_SUMMARY_JSON}")

if not figure_manifest["exists"].astype(bool).all():
    missing_figures = figure_manifest.loc[
        ~figure_manifest["exists"].astype(bool),
        "figure_path",
    ].tolist()
    raise RuntimeError(f"Missing branch contribution figures: {missing_figures}")

required_outputs = [
    BRANCH_FIGURE_PNG,
    BRANCH_FIGURE_PDF,
    BRANCH_FIGURE_SVG,
    BRANCH_PRIMARY_SUMMARY_CSV,
    BRANCH_FIGURE_MANIFEST_CSV,
    BRANCH_FIGURE_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 8 outputs: {missing_outputs}")

print("Status: PASS")

In [ ]:
# Cell 9: Final synthesis and status for Notebook 11

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))

try:
    RESULTS_11_DIR
except NameError:
    RESULTS_11_DIR = PROJECT_ROOT / "results" / "11_hybrid_anatomy_aware_classifier_training"

RESULTS_11_DIR.mkdir(parents=True, exist_ok=True)

HYBRID_EXPERIMENT_ID = "hybrid_full_image_plus_od_centered_2p5x_effb0"
HYBRID_MODEL_ID = "hybrid__full_image_plus_od_centered_2p5x_effb0"

HYBRID_TEST_EVAL_ROOT = RESULTS_11_DIR / "hybrid_locked_test_evaluation"
HYBRID_COMPARISON_ROOT = RESULTS_11_DIR / "hybrid_vs_internal_baselines_comparison"
BRANCH_ANALYSIS_ROOT = RESULTS_11_DIR / "hybrid_branch_contribution_analysis"
FIGURE_ROOT = RESULTS_11_DIR / "figures"

SETUP_SUMMARY_JSON = RESULTS_11_DIR / "notebook11_setup_summary.json"
RUNTIME_CONFIG_JSON = RESULTS_11_DIR / "notebook11_hybrid_runtime_config.json"
TRAINING_CONFIG_JSON = RESULTS_11_DIR / "notebook11_hybrid_training_config.json"

HYBRID_TRAINING_RESULT_CSV = (
    RESULTS_11_DIR
    / "hybrid_training_runs"
    / HYBRID_EXPERIMENT_ID
    / f"notebook11_training_result_{HYBRID_EXPERIMENT_ID}.csv"
)

HYBRID_TRAINING_PROGRESS_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_training_progress.csv"
)

HYBRID_VALIDATION_SELECTION_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

HYBRID_TEST_IMAGE_METRICS_CSV = (
    HYBRID_TEST_EVAL_ROOT
    / "notebook11_hybrid_test_image_level_metrics.csv"
)

HYBRID_TEST_PATIENT_METRICS_CSV = (
    HYBRID_TEST_EVAL_ROOT
    / "notebook11_hybrid_test_patient_level_metrics.csv"
)

HYBRID_TEST_EVALUATION_SUMMARY_JSON = (
    HYBRID_TEST_EVAL_ROOT
    / "notebook11_hybrid_locked_test_evaluation_summary.json"
)

HYBRID_BOOTSTRAP_CI_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_combined_internal_and_hybrid_bootstrap_ci.csv"
)

HYBRID_PAIRED_COMPARISON_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_paired_hybrid_vs_internal_baselines.csv"
)

HYBRID_INTERPRETATION_CSV = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_hybrid_vs_internal_baselines_interpretation.csv"
)

HYBRID_COMPARISON_SUMMARY_JSON = (
    HYBRID_COMPARISON_ROOT
    / "notebook11_hybrid_vs_internal_baselines_comparison_summary.json"
)

HYBRID_PRIMARY_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_primary_hybrid_vs_internal_baselines_summary.csv"
)

HYBRID_PRIMARY_DELTAS_CSV = (
    RESULTS_11_DIR
    / "notebook11_primary_hybrid_deltas_vs_internal_baselines.csv"
)

HYBRID_FIGURE_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_hybrid_comparison_figure_summary.json"
)

HYBRID_FIGURE_MANIFEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_hybrid_comparison_figure_manifest.csv"
)

BRANCH_PATIENT_METRICS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_patient_metrics.csv"
)

BRANCH_IMAGE_METRICS_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_image_metrics.csv"
)

BRANCH_PAIRED_BOOTSTRAP_CSV = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_paired_bootstrap.csv"
)

BRANCH_ANALYSIS_SUMMARY_JSON = (
    BRANCH_ANALYSIS_ROOT
    / "notebook11_hybrid_branch_contribution_summary.json"
)

BRANCH_PRIMARY_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_primary_summary.csv"
)

BRANCH_FIGURE_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_figure_summary.json"
)

BRANCH_FIGURE_MANIFEST_CSV = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_figure_manifest.csv"
)

MAIN_HYBRID_COMPARISON_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook11_main_hybrid_vs_internal_baselines_comparison.png"
)

MAIN_HYBRID_COMPARISON_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook11_main_hybrid_vs_internal_baselines_comparison.pdf"
)

MAIN_HYBRID_COMPARISON_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook11_main_hybrid_vs_internal_baselines_comparison.svg"
)

BRANCH_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook11_hybrid_branch_contribution_figure.png"
)

BRANCH_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook11_hybrid_branch_contribution_figure.pdf"
)

BRANCH_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook11_hybrid_branch_contribution_figure.svg"
)

NOTEBOOK11_FINAL_SYNTHESIS_CSV = (
    RESULTS_11_DIR
    / "notebook11_final_synthesis.csv"
)

NOTEBOOK11_FINAL_INTERPRETATION_CSV = (
    RESULTS_11_DIR
    / "notebook11_final_interpretation.csv"
)

NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_final_branch_contribution_summary.csv"
)

NOTEBOOK11_FINAL_CHECKS_CSV = (
    RESULTS_11_DIR
    / "notebook11_final_status_checks.csv"
)

NOTEBOOK11_FINAL_STATUS_JSON = (
    RESULTS_11_DIR
    / "notebook11_final_status.json"
)

NOTEBOOK11_FINAL_STATUS_TXT = (
    RESULTS_11_DIR
    / "notebook11_FINAL_STATUS.txt"
)

required_inputs = [
    SETUP_SUMMARY_JSON,
    RUNTIME_CONFIG_JSON,
    TRAINING_CONFIG_JSON,
    HYBRID_TRAINING_RESULT_CSV,
    HYBRID_TRAINING_PROGRESS_CSV,
    HYBRID_VALIDATION_SELECTION_JSON,
    HYBRID_TEST_IMAGE_METRICS_CSV,
    HYBRID_TEST_PATIENT_METRICS_CSV,
    HYBRID_TEST_EVALUATION_SUMMARY_JSON,
    HYBRID_BOOTSTRAP_CI_CSV,
    HYBRID_PAIRED_COMPARISON_CSV,
    HYBRID_INTERPRETATION_CSV,
    HYBRID_COMPARISON_SUMMARY_JSON,
    HYBRID_PRIMARY_SUMMARY_CSV,
    HYBRID_PRIMARY_DELTAS_CSV,
    HYBRID_FIGURE_SUMMARY_JSON,
    HYBRID_FIGURE_MANIFEST_CSV,
    BRANCH_PATIENT_METRICS_CSV,
    BRANCH_IMAGE_METRICS_CSV,
    BRANCH_PAIRED_BOOTSTRAP_CSV,
    BRANCH_ANALYSIS_SUMMARY_JSON,
    BRANCH_PRIMARY_SUMMARY_CSV,
    BRANCH_FIGURE_SUMMARY_JSON,
    BRANCH_FIGURE_MANIFEST_CSV,
    MAIN_HYBRID_COMPARISON_FIGURE_PNG,
    MAIN_HYBRID_COMPARISON_FIGURE_PDF,
    MAIN_HYBRID_COMPARISON_FIGURE_SVG,
    BRANCH_FIGURE_PNG,
    BRANCH_FIGURE_PDF,
    BRANCH_FIGURE_SVG,
]

input_audit = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "path": str(path),
            "exists": Path(path).exists(),
        }
        for path in required_inputs
    ]
)

missing_inputs = input_audit.loc[
    ~input_audit["exists"],
    "path",
].tolist()

if missing_inputs:
    print(input_audit.to_string(index=False))
    raise FileNotFoundError(f"Missing Notebook 11 final inputs: {missing_inputs}")

with open(SETUP_SUMMARY_JSON, "r", encoding="utf-8") as f:
    setup_summary = json.load(f)

with open(RUNTIME_CONFIG_JSON, "r", encoding="utf-8") as f:
    runtime_config = json.load(f)

with open(TRAINING_CONFIG_JSON, "r", encoding="utf-8") as f:
    training_config = json.load(f)

with open(HYBRID_VALIDATION_SELECTION_JSON, "r", encoding="utf-8") as f:
    validation_selection = json.load(f)

with open(HYBRID_TEST_EVALUATION_SUMMARY_JSON, "r", encoding="utf-8") as f:
    test_summary = json.load(f)

with open(HYBRID_COMPARISON_SUMMARY_JSON, "r", encoding="utf-8") as f:
    comparison_summary = json.load(f)

with open(HYBRID_FIGURE_SUMMARY_JSON, "r", encoding="utf-8") as f:
    hybrid_figure_summary = json.load(f)

with open(BRANCH_ANALYSIS_SUMMARY_JSON, "r", encoding="utf-8") as f:
    branch_analysis_summary = json.load(f)

with open(BRANCH_FIGURE_SUMMARY_JSON, "r", encoding="utf-8") as f:
    branch_figure_summary = json.load(f)

training_result = pd.read_csv(HYBRID_TRAINING_RESULT_CSV)
training_progress = pd.read_csv(HYBRID_TRAINING_PROGRESS_CSV)
test_image_metrics = pd.read_csv(HYBRID_TEST_IMAGE_METRICS_CSV)
test_patient_metrics = pd.read_csv(HYBRID_TEST_PATIENT_METRICS_CSV)
bootstrap_ci = pd.read_csv(HYBRID_BOOTSTRAP_CI_CSV)
paired_comparison = pd.read_csv(HYBRID_PAIRED_COMPARISON_CSV)
interpretation_df = pd.read_csv(HYBRID_INTERPRETATION_CSV)
primary_summary = pd.read_csv(HYBRID_PRIMARY_SUMMARY_CSV)
primary_deltas = pd.read_csv(HYBRID_PRIMARY_DELTAS_CSV)
branch_patient_metrics = pd.read_csv(BRANCH_PATIENT_METRICS_CSV)
branch_image_metrics = pd.read_csv(BRANCH_IMAGE_METRICS_CSV)
branch_bootstrap = pd.read_csv(BRANCH_PAIRED_BOOTSTRAP_CSV)
branch_primary_summary = pd.read_csv(BRANCH_PRIMARY_SUMMARY_CSV)
hybrid_figure_manifest = pd.read_csv(HYBRID_FIGURE_MANIFEST_CSV)
branch_figure_manifest = pd.read_csv(BRANCH_FIGURE_MANIFEST_CSV)

def metric_from_summary(model_id, metric):
    block = primary_summary[
        (primary_summary["model_id"].astype(str) == model_id)
        & (primary_summary["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(f"Metric row not found uniquely: {model_id} | {metric}")

    return block.iloc[0]


def branch_metric(branch_mode, metric):
    block = branch_patient_metrics[
        branch_patient_metrics["branch_mode"].astype(str) == branch_mode
    ]

    if len(block) != 1:
        raise RuntimeError(f"Branch metric row not found uniquely: {branch_mode}")

    return float(block[metric].iloc[0])


hybrid_auroc_row = metric_from_summary(HYBRID_MODEL_ID, "auroc")
hybrid_bacc_row = metric_from_summary(HYBRID_MODEL_ID, "balanced_accuracy")
hybrid_brier_row = metric_from_summary(HYBRID_MODEL_ID, "brier_score")

significant_brier_improvements = primary_deltas[
    (primary_deltas["metric"].astype(str) == "brier_score")
    & (primary_deltas["delta_hybrid_minus_reference"].astype(float) < 0)
    & (primary_deltas["delta_ci95_high"].astype(float) < 0)
].copy()

branch_key_rows = []

for branch_mode in [
    "original_hybrid",
    "full_image_branch_only",
    "od_centered_branch_only",
    "both_branches_neutralized",
]:
    branch_key_rows.append(
        {
            "branch_mode": branch_mode,
            "patient_auroc": branch_metric(branch_mode, "auroc"),
            "patient_balanced_accuracy": branch_metric(branch_mode, "balanced_accuracy"),
            "patient_brier_score": branch_metric(branch_mode, "brier_score"),
        }
    )

branch_key_df = pd.DataFrame(branch_key_rows)
branch_key_df.to_csv(NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV, index=False)

final_synthesis = pd.DataFrame(
    [
        {
            "item": "hybrid_locked_test_patient_performance",
            "finding": (
                "The hybrid full-image plus OD-centered 2.5x model achieved ceiling patient-level discrimination on the locked test set."
            ),
            "evidence": (
                f"AUROC={float(hybrid_auroc_row['observed']):.6f}; "
                f"balanced_accuracy={float(hybrid_bacc_row['observed']):.6f}; "
                f"Brier={float(hybrid_brier_row['observed']):.6f}."
            ),
            "journal_interpretation": (
                "The hybrid is a strong internal candidate, but discrimination is affected by ceiling effects and must be interpreted together with calibration, XAI, shortcut-control, and external validation."
            ),
        },
        {
            "item": "hybrid_probabilistic_advantage",
            "finding": (
                "The hybrid showed significant Brier-score improvement against selected internal baselines."
            ),
            "evidence": (
                "; ".join(
                    [
                        (
                            f"{row['reference_model_id']}: "
                            f"delta={float(row['delta_hybrid_minus_reference']):.6f}, "
                            f"CI=[{float(row['delta_ci95_low']):.6f}, {float(row['delta_ci95_high']):.6f}], "
                            f"p={float(row['paired_bootstrap_p_two_sided']):.4f}"
                        )
                        for _, row in significant_brier_improvements.iterrows()
                    ]
                )
                if len(significant_brier_improvements) > 0
                else "No significant Brier-score improvements detected."
            ),
            "journal_interpretation": (
                "The main internal advantage of the hybrid is probabilistic reliability rather than AUROC superiority, because multiple models are already near ceiling."
            ),
        },
        {
            "item": "branch_contribution",
            "finding": (
                "Branch-neutralization analysis showed complementary branch contribution, with the full-image branch carrying stronger standalone signal and the OD-centered branch improving the joint decision."
            ),
            "evidence": (
                f"Original hybrid Brier={branch_metric('original_hybrid', 'brier_score'):.6f}; "
                f"full-image-only Brier={branch_metric('full_image_branch_only', 'brier_score'):.6f}; "
                f"OD-centered-only Brier={branch_metric('od_centered_branch_only', 'brier_score'):.6f}; "
                f"both-neutralized AUROC={branch_metric('both_branches_neutralized', 'auroc'):.6f}."
            ),
            "journal_interpretation": (
                "The hybrid is not a trivial duplicate of one branch. However, it remains partly dependent on global full-image signal, so anatomical plausibility must still be audited with XAI and perturbation analyses."
            ),
        },
        {
            "item": "model_selection_policy",
            "finding": (
                "Notebook 11 does not finalize the clinical model. It provides a hybrid candidate for the next integrated internal audit."
            ),
            "evidence": (
                "The next notebook is defined as integrated internal reliability and anatomical audit before external validation."
            ),
            "journal_interpretation": (
                "The final model must be frozen only after jointly evaluating discrimination, calibration, patient consistency, anatomical plausibility, shortcut robustness, and branch contribution."
            ),
        },
    ]
)

final_synthesis.to_csv(NOTEBOOK11_FINAL_SYNTHESIS_CSV, index=False)

final_interpretation = pd.DataFrame(
    [
        {
            "section": "primary_claim",
            "text": (
                "The hybrid anatomy-aware classifier preserved ceiling-level internal discrimination while improving patient-level Brier score against several internal baselines."
            ),
        },
        {
            "section": "branch_claim",
            "text": (
                "Branch-neutralization showed that the full-image branch carries stronger standalone predictive signal, but the OD-centered branch contributes to the calibrated joint decision."
            ),
        },
        {
            "section": "caution",
            "text": (
                "The hybrid should not yet be declared the final model because internal HYGD performance is near ceiling and the outside-OD control also performs strongly."
            ),
        },
        {
            "section": "next_step",
            "text": (
                "Proceed to an integrated internal audit notebook combining performance, calibration, XAI, anatomical enrichment, ablation robustness, shortcut controls, branch contribution, and final model-freezing rules."
            ),
        },
    ]
)

final_interpretation.to_csv(NOTEBOOK11_FINAL_INTERPRETATION_CSV, index=False)

expected_models = {
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
    HYBRID_MODEL_ID,
}

expected_branch_modes = {
    "original_hybrid",
    "full_image_branch_only",
    "od_centered_branch_only",
    "both_branches_neutralized",
}

models_in_primary_summary = set(primary_summary["model_id"].astype(str))
branch_modes_in_metrics = set(branch_patient_metrics["branch_mode"].astype(str))

final_checks = pd.DataFrame(
    [
        {
            "check_name": "all_required_inputs_exist",
            "status": bool(input_audit["exists"].all()),
            "details": f"{int(input_audit['exists'].sum())}/{len(input_audit)} inputs found",
        },
        {
            "check_name": "hybrid_training_completed",
            "status": bool(training_progress["completed"].astype(bool).iloc[0]),
            "details": str(HYBRID_TRAINING_RESULT_CSV),
        },
        {
            "check_name": "hybrid_locked_test_metrics_exist",
            "status": bool(Path(HYBRID_TEST_PATIENT_METRICS_CSV).exists()),
            "details": str(HYBRID_TEST_PATIENT_METRICS_CSV),
        },
        {
            "check_name": "all_expected_models_in_hybrid_comparison",
            "status": expected_models.issubset(models_in_primary_summary),
            "details": ",".join(sorted(models_in_primary_summary)),
        },
        {
            "check_name": "hybrid_model_in_primary_summary",
            "status": HYBRID_MODEL_ID in models_in_primary_summary,
            "details": HYBRID_MODEL_ID,
        },
        {
            "check_name": "all_branch_modes_evaluated",
            "status": expected_branch_modes.issubset(branch_modes_in_metrics),
            "details": ",".join(sorted(branch_modes_in_metrics)),
        },
        {
            "check_name": "both_neutralized_negative_control_behaves_as_expected",
            "status": abs(branch_metric("both_branches_neutralized", "auroc") - 0.5) < 1e-8,
            "details": f"AUROC={branch_metric('both_branches_neutralized', 'auroc'):.6f}",
        },
        {
            "check_name": "hybrid_comparison_figures_exist",
            "status": bool(hybrid_figure_manifest["exists"].astype(bool).all()),
            "details": f"{int(hybrid_figure_manifest['exists'].astype(bool).sum())}/{len(hybrid_figure_manifest)} figures found",
        },
        {
            "check_name": "branch_contribution_figures_exist",
            "status": bool(branch_figure_manifest["exists"].astype(bool).all()),
            "details": f"{int(branch_figure_manifest['exists'].astype(bool).sum())}/{len(branch_figure_manifest)} figures found",
        },
        {
            "check_name": "next_notebook_defined",
            "status": True,
            "details": "12_integrated_internal_model_audit_and_freezing.ipynb",
        },
    ]
)

final_checks.to_csv(NOTEBOOK11_FINAL_CHECKS_CSV, index=False)

if not final_checks["status"].astype(bool).all():
    print(final_checks.to_string(index=False))
    raise RuntimeError("One or more Notebook 11 final checks failed.")

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, list):
        return [json_safe(v) for v in value]

    if isinstance(value, tuple):
        return [json_safe(v) for v in value]

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        value = float(value)
        return None if not np.isfinite(value) else value

    if isinstance(value, float):
        return None if not np.isfinite(value) else value

    if pd.isna(value) if not isinstance(value, (list, dict, tuple)) else False:
        return None

    return value

final_status = {
    "notebook": "11_hybrid_anatomy_aware_classifier_training",
    "status": "PASS",
    "analysis_scope": (
        "Hybrid full-image plus OD-centered classifier training, locked-test evaluation, internal baseline comparison, and branch contribution analysis."
    ),
    "hybrid_model_id": HYBRID_MODEL_ID,
    "hybrid_experiment_id": HYBRID_EXPERIMENT_ID,
    "full_branch_backbone": runtime_config.get("full_branch_backbone", "efficientnet_b0"),
    "anatomy_branch_backbone": runtime_config.get("anatomy_branch_backbone", "efficientnet_b0"),
    "fusion_strategy": runtime_config.get("fusion_strategy", "late_feature_concatenation"),
    "best_checkpoint": validation_selection.get("selected_best_checkpoint", ""),
    "best_epoch_one_indexed": validation_selection.get("selected_best_epoch_one_indexed", None),
    "locked_test_patient_auroc": float(hybrid_auroc_row["observed"]),
    "locked_test_patient_balanced_accuracy": float(hybrid_bacc_row["observed"]),
    "locked_test_patient_brier_score": float(hybrid_brier_row["observed"]),
    "significant_brier_improvements": significant_brier_improvements.to_dict(orient="records"),
    "branch_contribution_summary": branch_key_df.to_dict(orient="records"),
    "primary_interpretation": (
        "The hybrid model is a strong internal candidate: it preserves ceiling-level patient discrimination and improves Brier score against selected baselines. "
        "Branch analysis indicates complementary contribution, but the model remains partly dependent on full-image signal and must undergo integrated XAI, calibration, shortcut and external validation before final selection."
    ),
    "recommended_next_notebook": "12_integrated_internal_model_audit_and_freezing.ipynb",
    "recommended_next_step": (
        "Integrate all internal evidence from Notebooks 07-11, audit XAI/anatomical plausibility for the hybrid and selected comparators, then freeze the model set for external validation."
    ),
    "outputs": {
        "final_synthesis_csv": str(NOTEBOOK11_FINAL_SYNTHESIS_CSV),
        "final_interpretation_csv": str(NOTEBOOK11_FINAL_INTERPRETATION_CSV),
        "final_branch_summary_csv": str(NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV),
        "final_checks_csv": str(NOTEBOOK11_FINAL_CHECKS_CSV),
        "final_status_json": str(NOTEBOOK11_FINAL_STATUS_JSON),
        "final_status_txt": str(NOTEBOOK11_FINAL_STATUS_TXT),
        "hybrid_comparison_figure_png": str(MAIN_HYBRID_COMPARISON_FIGURE_PNG),
        "branch_contribution_figure_png": str(BRANCH_FIGURE_PNG),
    },
}

with open(NOTEBOOK11_FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(final_status), f, indent=2)

status_lines = [
    "Notebook 11 final status: PASS",
    "",
    "Scope:",
    final_status["analysis_scope"],
    "",
    "Hybrid model:",
    HYBRID_MODEL_ID,
    "",
    "Main result:",
    final_status["primary_interpretation"],
    "",
    "Locked-test patient-level results:",
    f"- AUROC: {float(hybrid_auroc_row['observed']):.6f}",
    f"- Balanced accuracy: {float(hybrid_bacc_row['observed']):.6f}",
    f"- Brier score: {float(hybrid_brier_row['observed']):.6f}",
    "",
    "Branch contribution summary:",
]

for _, row in branch_key_df.iterrows():
    status_lines.append(
        f"- {row['branch_mode']}: AUROC={row['patient_auroc']:.6f}, "
        f"BAcc={row['patient_balanced_accuracy']:.6f}, "
        f"Brier={row['patient_brier_score']:.6f}"
    )

status_lines.extend(
    [
        "",
        "Recommended next notebook:",
        "12_integrated_internal_model_audit_and_freezing.ipynb",
        "",
        f"Final synthesis CSV: {NOTEBOOK11_FINAL_SYNTHESIS_CSV}",
        f"Final interpretation CSV: {NOTEBOOK11_FINAL_INTERPRETATION_CSV}",
        f"Final branch summary CSV: {NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV}",
        f"Final checks CSV: {NOTEBOOK11_FINAL_CHECKS_CSV}",
        f"Final status JSON: {NOTEBOOK11_FINAL_STATUS_JSON}",
    ]
)

NOTEBOOK11_FINAL_STATUS_TXT.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

required_outputs = [
    NOTEBOOK11_FINAL_SYNTHESIS_CSV,
    NOTEBOOK11_FINAL_INTERPRETATION_CSV,
    NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV,
    NOTEBOOK11_FINAL_CHECKS_CSV,
    NOTEBOOK11_FINAL_STATUS_JSON,
    NOTEBOOK11_FINAL_STATUS_TXT,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Notebook 11 final outputs: {missing_outputs}")

print("Notebook 11 final synthesis:")
print(final_synthesis.to_string(index=False))

print("\nBranch contribution final summary:")
print(branch_key_df.to_string(index=False))

print("\nFinal interpretation:")
print(final_interpretation.to_string(index=False))

print("\nFinal checks:")
print(final_checks.to_string(index=False))

print(f"\nFinal synthesis CSV: {NOTEBOOK11_FINAL_SYNTHESIS_CSV}")
print(f"Final interpretation CSV: {NOTEBOOK11_FINAL_INTERPRETATION_CSV}")
print(f"Final branch summary CSV: {NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV}")
print(f"Final checks CSV: {NOTEBOOK11_FINAL_CHECKS_CSV}")
print(f"Final status JSON: {NOTEBOOK11_FINAL_STATUS_JSON}")
print(f"Final status TXT: {NOTEBOOK11_FINAL_STATUS_TXT}")

print("Status: PASS")